In [1]:
import subprocess
import librosa
import numpy as np
import json
from pytube import YouTube
import os
from fastdtw import fastdtw
import time
import re
import urllib.parse as urlparse

# Sample JSON timestamps for the first recording
timestamps_json = '''
[{"t":0,"mix":0},{"t":2.304,"mix":1},{"t":3.448,"mix":2},{"t":4.789,"mix":3},{"t":5.885,
"mix":4},{"t":7.175,"mix":5},{"t":8.245,"mix":6},{"t":9.22,"mix":7},{"t":10.675,"mix":8},
{"t":11.898,"mix":9},{"t":13.318,"mix":10},{"t":14.545,"mix":11},{"t":15.779,"mix":12},{"t":17.01,
"mix":13},{"t":18.512,"mix":14},{"t":19.772,"mix":15},{"t":21.002,"mix":16},{"t":22.171,"mix":17},
{"t":23.305,"mix":18},{"t":24.586,"mix":19},{"t":25.83,"mix":20},{"t":26.985,"mix":21},{"t":28.253,
"mix":22},{"t":29.598,"mix":23},{"t":30.858,"mix":24},{"t":31.939,"mix":25},{"t":33.173,"mix":26},
{"t":34.393,"mix":27},{"t":35.56,"mix":28},{"t":36.671,"mix":29},{"t":37.905,"mix":30},{"t":39.089,
"mix":31},{"t":40.277,"mix":32},{"t":41.378,"mix":33},{"t":42.421,"mix":34},{"t":43.685,"mix":35},
{"t":44.823,"mix":36},{"t":46.13,"mix":37},{"t":47.344,"mix":38},{"t":48.449,"mix":39},{"t":49.65,
"mix":40},{"t":50.883,"mix":41},{"t":52.107,"mix":42},{"t":53.341,"mix":43},{"t":54.631,"mix":44},
{"t":55.762,"mix":45},{"t":57.069,"mix":46},{"t":58.297,"mix":47},{"t":59.571,"mix":48},{"t":60.852,
"mix":49},{"t":62.015,"mix":50},{"t":63.22,"mix":51},{"t":64.571,"mix":52},{"t":65.799,"mix":53},
{"t":67.053,"mix":54},{"t":68.255,"mix":55},{"t":69.468,"mix":56},{"t":70.868,"mix":57},{"t":72.161,
"mix":58},{"t":73.364,"mix":59},{"t":74.476,"mix":60},{"t":75.688,"mix":61},{"t":76.924,"mix":62},
{"t":78.085,"mix":63},{"t":79.209,"mix":64},{"t":80.497,"mix":65},{"t":81.664,"mix":66},{"t":82.913,
"mix":67},{"t":83.967,"mix":68},{"t":85.273,"mix":69},{"t":86.669,"mix":70},{"t":87.689,"mix":71},
{"t":88.848,"mix":72},{"t":90.14,"mix":73},{"t":91.356,"mix":74},{"t":92.651,"mix":75},{"t":93.91,
"mix":76},{"t":95.046,"mix":77},{"t":96.207,"mix":78},{"t":97.467,"mix":79},{"t":98.726,"mix":80},
{"t":99.957,"mix":81},{"t":101.195,"mix":82},{"t":102.662,"mix":83},{"t":104.044,"mix":84},
{"t":105.606,"mix":85},{"t":107.506,"mix":86},{"t":108.687,"mix":87},{"t":110.131,"mix":88},
{"t":111.754,"mix":89},{"t":113.204,"mix":90},{"t":114.502,"mix":91},{"t":115.939,"mix":92},
{"t":117.74,"mix":93},{"t":119.654,"mix":94},{"t":121.432,"mix":95},{"t":122.893,"mix":96},
{"t":124.305,"mix":97},{"t":125.619,"mix":98},{"t":126.983,"mix":99},{"t":128.39,"mix":100},
{"t":129.54,"mix":101},{"t":130.841,"mix":102},{"t":132.213,"mix":103},{"t":133.504,"mix":104},
{"t":134.702,"mix":105},{"t":135.835,"mix":106},{"t":137.172,"mix":107},{"t":138.375,"mix":108},
{"t":139.679,"mix":109},{"t":140.995,"mix":110},{"t":142.17,"mix":111},{"t":143.38,"mix":112},
{"t":144.661,"mix":113},{"t":145.935,"mix":114},{"t":147.171,"mix":115},{"t":148.404,"mix":116},
{"t":149.572,"mix":117},{"t":150.851,"mix":118},{"t":152.125,"mix":119},{"t":153.416,"mix":120},
{"t":154.628,"mix":121},{"t":155.874,"mix":122},{"t":157.078,"mix":123},{"t":158.435,"mix":124},
{"t":159.649,"mix":125},{"t":161.041,"mix":126},{"t":162.35,"mix":127},{"t":163.742,"mix":128},
{"t":164.999,"mix":129},{"t":166.343,"mix":130},{"t":167.672,"mix":131},{"t":169.242,"mix":132},
{"t":170.595,"mix":133},{"t":171.967,"mix":134},{"t":173.269,"mix":135},{"t":174.769,"mix":136},
{"t":176.018,"mix":137},{"t":177.27,"mix":138},{"t":178.453,"mix":139},{"t":179.804,"mix":140},
{"t":181.058,"mix":141},{"t":182.292,"mix":142},{"t":183.58,"mix":143},{"t":184.884,"mix":144},
{"t":186.163,"mix":145},{"t":187.413,"mix":146},{"t":188.744,"mix":147},{"t":190.24,"mix":148},
{"t":191.86,"mix":149},{"t":193.18,"mix":150},{"t":194.578,"mix":151},{"t":196.06,"mix":152},
{"t":197.506,"mix":153},{"t":198.842,"mix":154},{"t":200.062,"mix":4},{"t":201.438,"mix":5},
{"t":202.581,"mix":6},{"t":203.881,"mix":7},{"t":205.184,"mix":8},{"t":206.562,"mix":9},{"t":207.771,
"mix":10},{"t":209.056,"mix":11},{"t":210.259,"mix":12},{"t":211.517,"mix":13},{"t":212.788,
"mix":14},{"t":214.154,"mix":15},{"t":215.413,"mix":16},{"t":216.615,"mix":17},{"t":217.967,
"mix":18},{"t":219.248,"mix":19},{"t":220.482,"mix":20},{"t":221.624,"mix":21},{"t":222.859,
"mix":22},{"t":224.229,"mix":23},{"t":225.518,"mix":24},{"t":226.746,"mix":25},{"t":228.025,
"mix":26},{"t":229.291,"mix":27},{"t":230.543,"mix":28},{"t":231.698,"mix":29},{"t":233.009,
"mix":30},{"t":234.204,"mix":31},{"t":235.454,"mix":32},{"t":236.742,"mix":33},{"t":237.927,
"mix":34},{"t":239.223,"mix":35},{"t":240.422,"mix":36},{"t":241.745,"mix":37},{"t":243.005,
"mix":38},{"t":244.167,"mix":39},{"t":245.455,"mix":40},{"t":246.712,"mix":41},{"t":247.956,
"mix":42},{"t":249.204,"mix":43},{"t":250.495,"mix":44},{"t":251.759,"mix":45},{"t":253.031,
"mix":46},{"t":254.314,"mix":47},{"t":255.53,"mix":48},{"t":256.831,"mix":49},{"t":258.095,
"mix":50},{"t":259.3,"mix":51},{"t":260.615,"mix":52},{"t":261.871,"mix":53},{"t":263.17,
"mix":54},{"t":264.399,"mix":55},{"t":265.653,"mix":56},{"t":267.036,"mix":57},{"t":268.376,
"mix":58},{"t":269.516,"mix":59},{"t":270.719,"mix":60},{"t":272.057,"mix":61},{"t":273.236,
"mix":62},{"t":274.388,"mix":63},{"t":275.614,"mix":64},{"t":276.902,"mix":65},{"t":278.057,
"mix":66},{"t":279.269,"mix":67},{"t":280.409,"mix":68},{"t":281.818,"mix":69},{"t":282.914,
"mix":70},{"t":284.061,"mix":71},{"t":285.333,"mix":72},{"t":286.643,"mix":73},{"t":287.972,
"mix":74},{"t":289.208,"mix":75},{"t":290.508,"mix":76},{"t":291.623,"mix":77},{"t":292.817,
"mix":78},{"t":294.09,"mix":79},{"t":295.32,"mix":80},{"t":296.52,"mix":81},{"t":297.764,
"mix":82},{"t":299.13,"mix":83},{"t":300.526,"mix":84},{"t":302.111,"mix":85},{"t":303.73,
"mix":86},{"t":305.056,"mix":87},{"t":306.517,"mix":88},{"t":308.177,"mix":89},{"t":309.514,
"mix":90},{"t":310.904,"mix":91},{"t":312.342,"mix":92},{"t":313.855,"mix":93},{"t":316.371,
"mix":94},{"t":317.684,"mix":95},{"t":319.121,"mix":96},{"t":320.451,"mix":97},{"t":321.77,
"mix":98},{"t":323.207,"mix":99},{"t":324.536,"mix":100},{"t":325.801,"mix":101},{"t":327.118,
"mix":102},{"t":328.442,"mix":103},{"t":329.769,"mix":104},{"t":330.996,"mix":105},{"t":332.234,
"mix":106},{"t":333.434,"mix":107},{"t":334.723,"mix":108},{"t":336.05,"mix":109},{"t":337.241,
"mix":110},{"t":338.434,"mix":111},{"t":339.674,"mix":112},{"t":340.907,"mix":113},{"t":342.179,
"mix":114},{"t":343.418,"mix":115},{"t":344.564,"mix":116},{"t":345.8,"mix":117},{"t":347.1,
"mix":118},{"t":348.279,"mix":119},{"t":349.653,"mix":120},{"t":350.797,"mix":121},{"t":352.018,
"mix":122},{"t":353.327,"mix":123},{"t":354.623,"mix":124},{"t":355.903,"mix":125},{"t":357.215,
"mix":126},{"t":358.486,"mix":127},{"t":359.85,"mix":128},{"t":361.126,"mix":129},{"t":362.403,
"mix":130},{"t":363.741,"mix":131},{"t":365.226,"mix":132},{"t":366.636,"mix":133},{"t":367.945,
"mix":134},{"t":369.286,"mix":135},{"t":370.636,"mix":136},{"t":371.85,"mix":137},{"t":373.106,
"mix":138},{"t":374.307,"mix":139},{"t":375.543,"mix":140},{"t":376.813,"mix":141},{"t":378.035,
"mix":142},{"t":379.342,"mix":143},{"t":380.542,"mix":144},{"t":381.86,"mix":145},{"t":383.069,
"mix":146},{"t":384.443,"mix":147},{"t":385.95,"mix":148},{"t":387.329,"mix":149},{"t":388.565,
"mix":150},{"t":389.932,"mix":155},{"t":391.392,"mix":156},{"t":393.076,"mix":157},{"t":394.489,
"mix":158},{"t":395.849,"mix":159},{"t":397.264,"mix":160},{"t":398.633,"mix":161},{"t":400.2,
"mix":162},{"t":401.643,"mix":163},{"t":403.066,"mix":164},{"t":404.474,"mix":165},{"t":405.79,
"mix":166},{"t":407.177,"mix":167},{"t":408.717,"mix":168},{"t":410.332,"mix":169},{"t":411.64,
"mix":170},{"t":412.986,"mix":171},{"t":414.145,"mix":172},{"t":415.385,"mix":173},{"t":416.641,
"mix":174},{"t":417.83,"mix":175},{"t":419.036,"mix":176},{"t":420.317,"mix":177},{"t":421.595,
"mix":178},{"t":422.853,"mix":179},{"t":424.076,"mix":180},{"t":425.229,"mix":181},{"t":426.62,
"mix":182},{"t":427.8,"mix":183},{"t":429.009,"mix":184},{"t":430.297,"mix":185},{"t":431.413,
"mix":186},{"t":432.841,"mix":187},{"t":434.037,"mix":188},{"t":435.341,"mix":189},{"t":436.625,
"mix":190},{"t":437.903,"mix":191},{"t":439.21,"mix":192},{"t":440.448,"mix":193},{"t":441.595,
"mix":194},{"t":442.907,"mix":195},{"t":444.305,"mix":196},{"t":445.626,"mix":197},{"t":446.924,
"mix":198},{"t":448.047,"mix":199},{"t":449.321,"mix":200},{"t":450.588,"mix":201},{"t":451.829,
"mix":202},{"t":452.929,"mix":203},{"t":454.176,"mix":204},{"t":455.41,"mix":205},{"t":456.632,
"mix":206},{"t":457.85,"mix":207},{"t":458.967,"mix":208},{"t":460.36,"mix":209},{"t":461.672,
"mix":210},{"t":463.119,"mix":211},{"t":464.26,"mix":212},{"t":465.571,"mix":213},{"t":466.825,
"mix":214},{"t":468.163,"mix":215},{"t":469.387,"mix":216},{"t":470.589,"mix":217},{"t":471.945,
"mix":218},{"t":473.202,"mix":219},{"t":474.424,"mix":220},{"t":475.597,"mix":221},{"t":476.963,
"mix":222},{"t":478.219,"mix":223},{"t":479.535,"mix":224},{"t":480.954,"mix":225},{"t":482.064,
"mix":226},{"t":483.303,"mix":227},{"t":484.551,"mix":228},{"t":485.795,"mix":229},{"t":486.888,
"mix":230},{"t":488.257,"mix":231},{"t":489.559,"mix":232},{"t":490.749,"mix":233},{"t":491.93,
"mix":234},{"t":493.154,"mix":235},{"t":494.448,"mix":236},{"t":495.704,"mix":237},{"t":496.923,
"mix":238},{"t":498.133,"mix":239},{"t":499.463,"mix":240},{"t":500.66,"mix":241},{"t":501.908,
"mix":242},{"t":503.09,"mix":243},{"t":504.355,"mix":244},{"t":505.557,"mix":245},{"t":506.736,
"mix":246},{"t":507.956,"mix":247},{"t":509.216,"mix":248},{"t":510.502,"mix":249},{"t":511.733,
"mix":250},{"t":513.08,"mix":251},{"t":514.313,"mix":252},{"t":515.646,"mix":253},{"t":516.938,
"mix":254},{"t":518.307,"mix":255},{"t":519.56,"mix":256},{"t":520.856,"mix":257},{"t":522.204,
"mix":258},{"t":523.478,"mix":259},{"t":524.721,"mix":260},{"t":526.163,"mix":261},{"t":527.541,
"mix":262},{"t":528.805,"mix":263},{"t":530.155,"mix":264},{"t":531.476,"mix":265},{"t":532.755,
"mix":266},{"t":534.051,"mix":267},{"t":535.31,"mix":268},{"t":536.688,"mix":269},{"t":538.029,
"mix":270},{"t":539.313,"mix":271},{"t":540.567,"mix":272},{"t":541.898,"mix":273},{"t":543.174,
"mix":274},{"t":544.518,"mix":275},{"t":545.845,"mix":276},{"t":547.171,"mix":277},{"t":548.517,
"mix":278},{"t":549.922,"mix":279},{"t":551.261,"mix":280},{"t":552.615,"mix":281},{"t":553.953,
"mix":282},{"t":555.667,"mix":283},{"t":557.246,"mix":284},{"t":558.607,"mix":285},{"t":560.033,
"mix":286},{"t":561.365,"mix":287},{"t":562.725,"mix":288},{"t":564.142,"mix":289},{"t":565.395,
"mix":290},{"t":566.817,"mix":291},{"t":568.06,"mix":292},{"t":569.432,"mix":293},{"t":570.849,
"mix":294},{"t":572.11,"mix":295},{"t":573.431,"mix":296},{"t":574.725,"mix":297},{"t":576.153,
"mix":298},{"t":577.37,"mix":299},{"t":578.618,"mix":300},{"t":579.902,"mix":301},{"t":581.223,
"mix":302},{"t":582.606,"mix":303},{"t":583.936,"mix":304},{"t":585.105,"mix":305},{"t":586.27,
"mix":306},{"t":587.573,"mix":307},{"t":588.733,"mix":308},{"t":590.09,"mix":309},{"t":591.244,
"mix":310},{"t":592.486,"mix":311},{"t":593.752,"mix":312},{"t":595.111,"mix":313},{"t":596.231,
"mix":314},{"t":597.479,"mix":315},{"t":598.764,"mix":316},{"t":600.055,"mix":317},{"t":601.315,
"mix":318},{"t":602.481,"mix":319},{"t":603.638,"mix":320},{"t":604.932,"mix":321},{"t":606.124,
"mix":322},{"t":607.335,"mix":323},{"t":608.575,"mix":324},{"t":609.899,"mix":325},{"t":611.27,
"mix":326},{"t":612.594,"mix":327},{"t":613.824,"mix":328},{"t":614.927,"mix":329},{"t":616.157,
"mix":330},{"t":617.331,"mix":331},{"t":618.79,"mix":332},{"t":620.174,"mix":333},{"t":621.572,
"mix":334},{"t":622.66,"mix":335},{"t":624.145,"mix":336},{"t":625.357,"mix":337},{"t":626.892,
"mix":338},{"t":628.084,"mix":339},{"t":629.441,"mix":340},{"t":630.881,"mix":341},{"t":632.157,
"mix":342},{"t":633.432,"mix":343},{"t":634.696,"mix":344},{"t":636.029,"mix":345},{"t":637.373,
"mix":346},{"t":638.667,"mix":347},{"t":639.934,"mix":348},{"t":641.234,"mix":349},{"t":642.646,
"mix":350},{"t":643.839,"mix":351},{"t":645.165,"mix":352},{"t":646.399,"mix":353},{"t":647.604,
"mix":354},{"t":648.896,"mix":355},{"t":650.237,"mix":356},{"t":651.471,"mix":357},{"t":652.777,
"mix":358},{"t":653.996,"mix":359},{"t":655.304,"mix":360},{"t":656.552,"mix":361},{"t":657.782,
"mix":362},{"t":659.099,"mix":363},{"t":660.435,"mix":364},{"t":661.631,"mix":365},{"t":663.007,
"mix":366},{"t":664.272,"mix":367},{"t":665.536,"mix":368},{"t":666.794,"mix":369},{"t":668.16,
"mix":370},{"t":669.498,"mix":371},{"t":670.906,"mix":372},{"t":672.395,"mix":373},{"t":673.775,
"mix":374},{"t":675.126,"mix":375},{"t":676.64,"mix":376},{"t":678.167,"mix":377},{"t":679.447,
"mix":378},{"t":680.753,"mix":379},{"t":682.057,"mix":380},{"t":683.399,"mix":381},{"t":684.864,
"mix":382},{"t":686.429,"mix":383},{"t":687.87,"mix":384},{"t":689.238,"mix":385},{"t":690.686,
"mix":386},{"t":691.973,"mix":387},{"t":693.313,"mix":388},{"t":694.696,"mix":389},{"t":696.082,
"mix":390},{"t":697.354,"mix":391},{"t":698.705,"mix":392},{"t":700.055,"mix":393},{"t":701.42,
"mix":394},{"t":702.694,"mix":395},{"t":704.098,"mix":396},{"t":705.286,"mix":397},{"t":706.778,
"mix":398},{"t":708.239,"mix":399},{"t":709.426,"mix":400},{"t":710.844,"mix":401},{"t":712.209,
"mix":402},{"t":713.433,"mix":403},{"t":714.572,"mix":404},{"t":715.714,"mix":405},{"t":716.914,
"mix":406},{"t":718.277,"mix":407},{"t":719.547,"mix":408},{"t":720.804,"mix":409},{"t":722.071,
"mix":410},{"t":723.281,"mix":411},{"t":724.632,"mix":412},{"t":725.829,"mix":413},{"t":727.081,
"mix":414},{"t":728.324,"mix":415},{"t":729.572,"mix":416},{"t":730.81,"mix":417},{"t":731.957,
"mix":418},{"t":733.237,"mix":419},{"t":734.615,"mix":420},{"t":736.034,"mix":421},{"t":737.267,
"mix":422},{"t":738.414,"mix":423},{"t":739.619,"mix":424},{"t":740.858,"mix":425},{"t":742.15,
"mix":426},{"t":743.346,"mix":427},{"t":744.671,"mix":428},{"t":745.999,"mix":429},{"t":747.162,
"mix":430},{"t":748.332,"mix":431},{"t":749.571,"mix":432},{"t":750.83,"mix":433},{"t":752.145,
"mix":434},{"t":753.309,"mix":435},{"t":754.504,"mix":436},{"t":755.78,"mix":437},{"t":756.951,
"mix":438},{"t":758.324,"mix":439},{"t":759.57,"mix":440},{"t":760.781,"mix":441},{"t":761.97,
"mix":442},{"t":763.276,"mix":443},{"t":764.546,"mix":444},{"t":765.814,"mix":445},{"t":767.013,
"mix":446},{"t":768.315,"mix":447},{"t":769.515,"mix":448},{"t":770.781,"mix":449},{"t":772.022,
"mix":450},{"t":773.269,"mix":451},{"t":774.476,"mix":452},{"t":775.754,"mix":453},{"t":776.968,
"mix":454},{"t":778.256,"mix":455},{"t":779.581,"mix":456},{"t":780.807,"mix":457},{"t":782.05,
"mix":458},{"t":783.283,"mix":459},{"t":784.601,"mix":460},{"t":785.844,"mix":461},{"t":787.137,
"mix":462},{"t":788.37,"mix":463},{"t":789.797,"mix":464},{"t":791.076,"mix":465},{"t":792.271,
"mix":466},{"t":793.471,"mix":467},{"t":794.613,"mix":468},{"t":796.002,"mix":469},{"t":797.134,
"mix":470},{"t":798.442,"mix":471},{"t":799.635,"mix":472},{"t":800.79,"mix":473},{"t":802.071,
"mix":474},{"t":803.186,"mix":475},{"t":804.476,"mix":476},{"t":805.648,"mix":477},{"t":806.75,
"mix":478},{"t":807.937,"mix":479},{"t":809.324,"mix":480},{"t":810.534,"mix":481},{"t":811.76,
"mix":482},{"t":813.037,"mix":483},{"t":814.227,"mix":484},{"t":815.393,"mix":485},{"t":816.698,
"mix":486},{"t":817.933,"mix":487},{"t":819.206,"mix":488},{"t":820.378,"mix":489},{"t":821.697,
"mix":490},{"t":823.117,"mix":491},{"t":824.722,"mix":492},{"t":826.476,"mix":493},{"t":827.777,
"mix":494},{"t":829.223,"mix":495},{"t":830.677,"mix":496},{"t":832.225,"mix":497},{"t":833.622,
"mix":498},{"t":835.04,"mix":499},{"t":836.445,"mix":500},{"t":838.721,"mix":501},{"t":840.35,
"mix":502},{"t":841.621,"mix":503},{"t":843.113,"mix":504},{"t":844.623,"mix":505},{"t":845.98,
"mix":506},{"t":847.448,"mix":507},{"t":848.663,"mix":508},{"t":849.997,"mix":509},{"t":851.326,
"mix":510},{"t":852.597,"mix":511},{"t":853.762,"mix":512},{"t":855.007,"mix":513},{"t":856.168,
"mix":514},{"t":857.401,"mix":515},{"t":858.762,"mix":516},{"t":860.012,"mix":517},{"t":861.19,
"mix":518},{"t":862.349,"mix":519},{"t":863.657,"mix":520},{"t":864.893,"mix":521},{"t":866.145,
"mix":522},{"t":867.415,"mix":523},{"t":868.699,"mix":524},{"t":869.888,"mix":525},{"t":871.076,
"mix":526},{"t":872.32,"mix":527},{"t":873.493,"mix":528},{"t":874.727,"mix":529},{"t":875.977,
"mix":530},{"t":877.274,"mix":531},{"t":878.524,"mix":532},{"t":879.729,"mix":533},{"t":881.021,
"mix":534},{"t":882.321,"mix":535},{"t":883.65,"mix":536},{"t":884.889,"mix":537},{"t":886.266,
"mix":538},{"t":887.765,"mix":539},{"t":889.143,"mix":540},{"t":890.562,"mix":541},{"t":891.922,
"mix":542},{"t":893.346,"mix":543},{"t":894.61,"mix":544},{"t":895.878,"mix":545},{"t":897.004,
"mix":546},{"t":898.314,"mix":547},{"t":899.534,"mix":548},{"t":900.911,"mix":549},{"t":902.169,
"mix":550},{"t":903.355,"mix":551},{"t":904.685,"mix":552},{"t":905.92,"mix":553},{"t":907.351,
"mix":554},{"t":908.975,"mix":555},{"t":910.221,"mix":556},{"t":911.74,"mix":557},{"t":913.093,
"mix":558},{"t":914.849,"mix":559},{"t":916.704,"mix":560},{"t":918.538,"mix":561},{"t":919.915,
"mix":562},{"t":921.535,"mix":563},{"t":923.272,"mix":564},{"t":924.639,"mix":565},{"t":925.873,
"mix":566},{"t":927.217,"mix":567},{"t":928.64,"mix":568},{"t":929.842,"mix":569},{"t":931.133,
"mix":570},{"t":932.32,"mix":571},{"t":933.558,"mix":572},{"t":934.971,"mix":573},{"t":936.239,
"mix":574},{"t":937.622,"mix":575},{"t":938.925,"mix":576},{"t":940.089,"mix":577},{"t":941.456,
"mix":578},{"t":942.697,"mix":579},{"t":944.116,"mix":580},{"t":945.284,"mix":581},{"t":946.714,
"mix":582},{"t":947.988,"mix":583},{"t":949.197,"mix":584},{"t":950.367,"mix":585},{"t":951.564,
"mix":586},{"t":952.77,"mix":587},{"t":953.967,"mix":588},{"t":955.38,"mix":589},{"t":956.599,
"mix":590},{"t":958.014,"mix":591},{"t":959.078,"mix":592},{"t":960.326,"mix":593},{"t":961.664,
"mix":594},{"t":962.987,"mix":595},{"t":964.267,"mix":596},{"t":965.563,"mix":597},{"t":966.81,
"mix":598},{"t":968.108,"mix":599},{"t":969.385,"mix":600},{"t":970.77,"mix":601},{"t":971.95,
"mix":602},{"t":973.294,"mix":603},{"t":974.641,"mix":604},{"t":975.859,"mix":605},{"t":977.141,
"mix":606},{"t":978.645,"mix":607},{"t":979.894,"mix":608},{"t":981.284,"mix":609},{"t":982.536,
"mix":610},{"t":983.875,"mix":611},{"t":985.147,"mix":612},{"t":986.445,"mix":613},{"t":987.682,
"mix":614},{"t":988.991,"mix":615},{"t":990.301,"mix":616},{"t":991.478,"mix":617},{"t":992.765,
"mix":618},{"t":993.984,"mix":619},{"t":995.257,"mix":620},{"t":996.492,"mix":621},{"t":997.766,
"mix":622},{"t":999.071,"mix":623},{"t":1000.294,"mix":624},{"t":1001.598,"mix":625},{"t":1002.891,
"mix":626},{"t":1004.209,"mix":627},{"t":1005.52,"mix":628},{"t":1006.778,"mix":629},{"t":1008.192,
"mix":630},{"t":1009.428,"mix":631},{"t":1010.733,"mix":632},{"t":1011.935,"mix":633},{"t":1013.317,
"mix":634},{"t":1014.63,"mix":635},{"t":1015.808,"mix":636},{"t":1017.047,"mix":637},{"t":1018.345,
"mix":638},{"t":1019.748,"mix":639},{"t":1020.999,"mix":640},{"t":1022.16,"mix":641},{"t":1023.353,
"mix":642},{"t":1024.639,"mix":643},{"t":1025.883,"mix":644},{"t":1027.107,"mix":645},{"t":1028.344,
"mix":646},{"t":1029.652,"mix":647},{"t":1030.772,"mix":648},{"t":1032.077,"mix":649},{"t":1033.359,
"mix":650},{"t":1034.582,"mix":651},{"t":1035.762,"mix":652},{"t":1036.991,"mix":653},{"t":1038.185,
"mix":654},{"t":1039.446,"mix":655},{"t":1040.668,"mix":656},{"t":1041.833,"mix":657},{"t":1043.068,
"mix":658},{"t":1044.372,"mix":659},{"t":1045.677,"mix":660},{"t":1046.901,"mix":661},{"t":1048.169,
"mix":662},{"t":1049.372,"mix":663},{"t":1050.582,"mix":664},{"t":1051.812,"mix":665},{"t":1053.037,
"mix":666},{"t":1054.28,"mix":667},{"t":1055.512,"mix":668},{"t":1056.76,"mix":669},{"t":1057.89,
"mix":670},{"t":1059.107,"mix":671},{"t":1060.32,"mix":672},{"t":1061.516,"mix":673},{"t":1062.758,
"mix":674},{"t":1064.005,"mix":675},{"t":1065.165,"mix":676},{"t":1066.493,"mix":677},{"t":1067.765,
"mix":678},{"t":1069.064,"mix":679},{"t":1070.34,"mix":680},{"t":1071.532,"mix":681},{"t":1072.814,
"mix":682},{"t":1074.018,"mix":683},{"t":1075.267,"mix":684},{"t":1076.556,"mix":685},{"t":1077.782,
"mix":686},{"t":1079.096,"mix":687},{"t":1080.317,"mix":688},{"t":1081.627,"mix":689},{"t":1082.89,
"mix":690},{"t":1084.216,"mix":691},{"t":1085.336,"mix":692},{"t":1086.782,"mix":693},{"t":1088.393,
"mix":694},{"t":1114.073,"mix":695},{"t":1117.335,"mix":696},{"t":1120.84,"mix":697},{"t":1124.702,
"mix":698},{"t":1128.432,"mix":699},{"t":1133.163,"mix":700},{"t":1136.765,"mix":701},{"t":1140.676,
"mix":702},{"t":1144.365,"mix":703},{"t":1148.199,"mix":704},{"t":1151.947,"mix":705},{"t":1155.524,
"mix":706},{"t":1159.205,"mix":707},{"t":1163.39,"mix":708},{"t":1167.228,"mix":709},{"t":1170.907,
"mix":710},{"t":1174.302,"mix":711},{"t":1178.535,"mix":712},{"t":1182.568,"mix":713},{"t":1186.303,
"mix":714},{"t":1189.98,"mix":715},{"t":1193.908,"mix":716},{"t":1198.34,"mix":717},{"t":1202.474,
"mix":718},{"t":1206.215,"mix":719},{"t":1209.767,"mix":720},{"t":1213.574,"mix":721},{"t":1217.635,
"mix":722},{"t":1221.551,"mix":723},{"t":1225.47,"mix":724},{"t":1229.741,"mix":725},{"t":1233.939,
"mix":726},{"t":1237.58,"mix":727},{"t":1241.451,"mix":728},{"t":1245.049,"mix":729},{"t":1249.121,
"mix":730},{"t":1252.911,"mix":731},{"t":1256.612,"mix":732},{"t":1260.293,"mix":733},{"t":1264.014,
"mix":734},{"t":1267.704,"mix":735},{"t":1271.551,"mix":736},{"t":1275.178,"mix":737},{"t":1278.984,
"mix":738},{"t":1282.782,"mix":739},{"t":1286.291,"mix":740},{"t":1289.997,"mix":741},{"t":1293.704,
"mix":742},{"t":1297.403,"mix":743},{"t":1300.755,"mix":744},{"t":1304.553,"mix":745},{"t":1309.165,
"mix":746},{"t":1312.614,"mix":747},{"t":1316.079,"mix":748},{"t":1319.796,"mix":749},{"t":1323.557,
"mix":750},{"t":1327.276,"mix":751},{"t":1331.02,"mix":752},{"t":1334.779,"mix":753},{"t":1338.714,
"mix":754},{"t":1342.654,"mix":755},{"t":1346.348,"mix":756},{"t":1350.058,"mix":757},{"t":1353.892,
"mix":758},{"t":1357.79,"mix":759},{"t":1361.931,"mix":760},{"t":1365.943,"mix":761},{"t":1369.724,
"mix":762},{"t":1373.776,"mix":763},{"t":1378.887,"mix":764},{"t":1382.692,"mix":765},{"t":1386.198,
"mix":766},{"t":1389.746,"mix":767},{"t":1393.276,"mix":768},{"t":1396.833,"mix":769},{"t":1400.444,
"mix":770},{"t":1404.058,"mix":771},{"t":1407.736,"mix":772},{"t":1411.133,"mix":773},{"t":1414.862,
"mix":774},{"t":1419.636,"mix":775},{"t":1423.313,"mix":776},{"t":1426.96,"mix":777},{"t":1430.395,
"mix":778},{"t":1433.94,"mix":779},{"t":1437.535,"mix":780},{"t":1441.033,"mix":781},{"t":1444.577,
"mix":782},{"t":1447.92,"mix":783},{"t":1451.295,"mix":784},{"t":1454.809,"mix":785},{"t":1458.513,
"mix":786},{"t":1461.928,"mix":787},{"t":1465.439,"mix":788},{"t":1469.048,"mix":789},{"t":1472.455,
"mix":790},{"t":1475.928,"mix":791},{"t":1479.4,"mix":792},{"t":1482.962,"mix":793},{"t":1486.621,
"mix":794},{"t":1490.084,"mix":795},{"t":1493.83,"mix":796},{"t":1498.401,"mix":797},{"t":1501.929,
"mix":798},{"t":1506.57,"mix":799},{"t":1511.3,"mix":800},{"t":1514.963,"mix":801},{"t":1518.983,
"mix":802},{"t":1522.845,"mix":803},{"t":1527.169,"mix":804},{"t":1531.414,"mix":805},{"t":1535.251,
"mix":806},{"t":1539.199,"mix":807},{"t":1542.742,"mix":808},{"t":1546.465,"mix":809},{"t":1549.872,
"mix":810},{"t":1552.84,"mix":811},{"t":1555.815,"mix":812},{"t":1558.73,"mix":813},{"t":1561.503,
"mix":814},{"t":1564.376,"mix":815},{"t":1567.188,"mix":816},{"t":1570.064,"mix":817},{"t":1572.823,
"mix":818},{"t":1575.642,"mix":819},{"t":1578.47,"mix":820},{"t":1581.283,"mix":821},{"t":1584.286,
"mix":822},{"t":1587.171,"mix":823},{"t":1590.247,"mix":824},{"t":1593.266,"mix":825},{"t":1596.158,
"mix":826},{"t":1599.002,"mix":827},{"t":1602.003,"mix":828},{"t":1604.988,"mix":829},{"t":1607.825,
"mix":830},{"t":1610.696,"mix":831},{"t":1613.7,"mix":832},{"t":1616.609,"mix":833},{"t":1619.455,
"mix":834},{"t":1622.709,"mix":835},{"t":1625.626,"mix":836},{"t":1628.401,"mix":837},{"t":1631.301,
"mix":838},{"t":1634.236,"mix":839},{"t":1637.043,"mix":840},{"t":1640.134,"mix":841},{"t":1642.964,
"mix":842},{"t":1645.877,"mix":843},{"t":1648.698,"mix":844},{"t":1651.912,"mix":845},{"t":1655.476,
"mix":846},{"t":1659.455,"mix":847},{"t":1663.565,"mix":848},{"t":1668.85,"mix":849},{"t":1672.769,
"mix":850},{"t":1676.576,"mix":851},{"t":1680.728,"mix":852},{"t":1684.177,"mix":853},{"t":1687.321,
"mix":854},{"t":1690.747,"mix":855},{"t":1693.794,"mix":856},{"t":1696.811,"mix":857},{"t":1699.811,
"mix":858},{"t":1702.662,"mix":859},{"t":1705.677,"mix":860},{"t":1708.788,"mix":861},{"t":1711.703,
"mix":862},{"t":1714.9,"mix":863},{"t":1718.051,"mix":864},{"t":1721.23,"mix":865},{"t":1724.524,
"mix":866},{"t":1727.843,"mix":867},{"t":1731.022,"mix":868},{"t":1734.393,"mix":869},{"t":1737.637,
"mix":870},{"t":1741.072,"mix":871},{"t":1744.374,"mix":872},{"t":1747.888,"mix":873},{"t":1751.444,
"mix":874},{"t":1755.176,"mix":875},{"t":1759.584,"mix":876},{"t":1763.913,"mix":877},{"t":1768.237,
"mix":878},{"t":1772.567,"mix":879},{"t":1777.291,"mix":880},{"t":1780.998,"mix":881},{"t":1785.374,
"mix":882},{"t":1788.901,"mix":883},{"t":1792.403,"mix":884},{"t":1795.817,"mix":885},{"t":1799.569,
"mix":886},{"t":1803.238,"mix":887},{"t":1806.768,"mix":888},{"t":1810.319,"mix":889},{"t":1814.382,
"mix":890},{"t":1817.932,"mix":891},{"t":1821.421,"mix":892},{"t":1825.089,"mix":893},{"t":1828.796,
"mix":894},{"t":1832.685,"mix":895},{"t":1836.46,"mix":896},{"t":1840.187,"mix":897},{"t":1844.06,
"mix":898},{"t":1847.85,"mix":899},{"t":1851.399,"mix":900},{"t":1855.348,"mix":901},{"t":1859.021,
"mix":902},{"t":1862.914,"mix":903},{"t":1867.256,"mix":904},{"t":1870.827,"mix":905},{"t":1874.574,
"mix":906},{"t":1878.669,"mix":907},{"t":1883.662,"mix":908},{"t":1887.052,"mix":909},{"t":1890.28,
"mix":910},{"t":1893.703,"mix":911},{"t":1897.087,"mix":912},{"t":1900.625,"mix":913},{"t":1904,
"mix":914},{"t":1907.481,"mix":915},{"t":1911.163,"mix":916},{"t":1914.761,"mix":917},{"t":1918.753,
"mix":918},{"t":1922.954,"mix":919},{"t":1927.129,"mix":920},{"t":1930.858,"mix":921},{"t":1934.64,
"mix":922},{"t":1940.06,"mix":923},{"t":1945.02,"mix":924},{"t":1948.523,"mix":925},{"t":1952.484,
"mix":926},{"t":1957.412,"mix":927},{"t":1961.655,"mix":928},{"t":1967.057,"mix":929},{"t":1971.512,
"mix":930},{"t":1976.145,"mix":931},{"t":1980.475,"mix":932},{"t":1985.18,"mix":933},{"t":1990.454,
"mix":934},{"t":1995.123,"mix":935},{"t":1999.295,"mix":936},{"t":2003.613,"mix":937},{"t":2008.661,
"mix":938},{"t":2013.311,"mix":939},{"t":2017.668,"mix":940},{"t":2023.21,"mix":941},{"t":2030.184,
"mix":942},{"t":2052.503,"mix":943},{"t":2053.24,"mix":944},{"t":2053.733,"mix":945},{"t":2054.255,
"mix":946},{"t":2054.756,"mix":947},{"t":2055.268,"mix":948},{"t":2055.782,"mix":949},{"t":2056.308,
"mix":950},{"t":2056.799,"mix":951},{"t":2057.338,"mix":952},{"t":2057.829,"mix":953},{"t":2058.372,
"mix":954},{"t":2058.894,"mix":955},{"t":2059.383,"mix":956},{"t":2059.906,"mix":957},{"t":2060.492,
"mix":958},{"t":2060.984,"mix":959},{"t":2061.541,"mix":960},{"t":2062.021,"mix":961},{"t":2062.587,
"mix":962},{"t":2063.096,"mix":963},{"t":2063.609,"mix":964},{"t":2064.152,"mix":965},{"t":2064.698,
"mix":966},{"t":2065.192,"mix":967},{"t":2065.705,"mix":968},{"t":2066.219,"mix":969},{"t":2066.756,
"mix":970},{"t":2067.303,"mix":971},{"t":2067.819,"mix":972},{"t":2068.396,"mix":973},{"t":2068.945,
"mix":974},{"t":2069.534,"mix":975},{"t":2070.055,"mix":976},{"t":2070.59,"mix":977},{"t":2071.12,
"mix":978},{"t":2071.648,"mix":979},{"t":2072.109,"mix":980},{"t":2072.694,"mix":981},{"t":2073.194,
"mix":982},{"t":2073.743,"mix":983},{"t":2074.298,"mix":984},{"t":2074.866,"mix":985},{"t":2075.377,
"mix":986},{"t":2075.917,"mix":987},{"t":2076.47,"mix":988},{"t":2077.03,"mix":989},{"t":2077.535,
"mix":990},{"t":2078.093,"mix":991},{"t":2078.631,"mix":992},{"t":2079.205,"mix":993},{"t":2079.777,
"mix":994},{"t":2080.318,"mix":995},{"t":2080.841,"mix":996},{"t":2081.418,"mix":997},{"t":2081.94,
"mix":998},{"t":2082.48,"mix":999},{"t":2083.023,"mix":1000},{"t":2083.563,"mix":1001},{"t":2084.055,
"mix":1002},{"t":2084.594,"mix":1003},{"t":2085.132,"mix":1004},{"t":2085.654,"mix":1005},
{"t":2086.183,"mix":1006},{"t":2086.757,"mix":1007},{"t":2087.192,"mix":1008},{"t":2087.749,
"mix":1009},{"t":2088.325,"mix":1010},{"t":2088.868,"mix":1011},{"t":2089.413,"mix":1012},
{"t":2089.968,"mix":1013},{"t":2090.522,"mix":1014},{"t":2091.064,"mix":1015},{"t":2091.564,
"mix":1016},{"t":2092.18,"mix":1017},{"t":2092.665,"mix":1018},{"t":2093.243,"mix":1019},
{"t":2093.753,"mix":1020},{"t":2094.34,"mix":1021},{"t":2094.856,"mix":1022},{"t":2095.391,
"mix":1023},{"t":2095.91,"mix":1024},{"t":2096.434,"mix":1025},{"t":2096.918,"mix":1026},
{"t":2097.476,"mix":1027},{"t":2098.006,"mix":1028},{"t":2098.553,"mix":1029},{"t":2099.083,
"mix":1030},{"t":2099.583,"mix":1031},{"t":2100.154,"mix":1032},{"t":2100.688,"mix":1033},
{"t":2101.199,"mix":1034},{"t":2101.688,"mix":1035},{"t":2102.279,"mix":1036},{"t":2102.781,
"mix":1037},{"t":2103.328,"mix":1038},{"t":2103.799,"mix":1039},{"t":2104.367,"mix":1040},
{"t":2104.968,"mix":1041},{"t":2105.467,"mix":1042},{"t":2106.032,"mix":1043},{"t":2106.531,
"mix":1044},{"t":2107.117,"mix":1045},{"t":2107.663,"mix":1046},{"t":2108.15,"mix":1047},
{"t":2108.672,"mix":1048},{"t":2109.275,"mix":1049},{"t":2109.783,"mix":1050},{"t":2110.291,
"mix":1051},{"t":2110.778,"mix":1052},{"t":2111.39,"mix":1053},{"t":2111.878,"mix":1054},
{"t":2112.429,"mix":1055},{"t":2112.98,"mix":1056},{"t":2113.489,"mix":1057},{"t":2113.99,
"mix":1058},{"t":2114.504,"mix":1059},{"t":2115.066,"mix":1060},{"t":2115.554,"mix":1061},
{"t":2116.043,"mix":1062},{"t":2116.633,"mix":1063},{"t":2117.19,"mix":1064},{"t":2117.675,
"mix":1065},{"t":2118.162,"mix":1066},{"t":2118.685,"mix":1067},{"t":2119.208,"mix":1068},
{"t":2119.737,"mix":1069},{"t":2120.197,"mix":1070},{"t":2120.771,"mix":1071},{"t":2121.301,
"mix":1072},{"t":2121.844,"mix":1073},{"t":2122.384,"mix":1074},{"t":2122.983,"mix":1075},
{"t":2123.482,"mix":1076},{"t":2124.026,"mix":1077},{"t":2124.546,"mix":1078},{"t":2125.136,
"mix":1079},{"t":2125.668,"mix":1080},{"t":2126.197,"mix":1081},{"t":2126.738,"mix":1082},
{"t":2127.325,"mix":1083},{"t":2127.863,"mix":1084},{"t":2128.432,"mix":1085},{"t":2128.919,
"mix":1086},{"t":2129.494,"mix":1087},{"t":2130.034,"mix":1088},{"t":2130.547,"mix":1089},
{"t":2131.084,"mix":1090},{"t":2131.693,"mix":1091},{"t":2132.222,"mix":1092},{"t":2132.748,
"mix":1093},{"t":2133.314,"mix":1094},{"t":2133.868,"mix":1095},{"t":2134.388,"mix":1096},
{"t":2134.924,"mix":1097},{"t":2135.436,"mix":1098},{"t":2135.957,"mix":1099},{"t":2136.449,
"mix":1100},{"t":2136.978,"mix":1101},{"t":2137.56,"mix":1102},{"t":2138.059,"mix":1103},
{"t":2138.608,"mix":1104},{"t":2139.118,"mix":1105},{"t":2139.621,"mix":1106},{"t":2140.215,
"mix":1107},{"t":2140.744,"mix":1108},{"t":2141.243,"mix":1109},{"t":2141.811,"mix":974},
{"t":2142.409,"mix":975},{"t":2142.959,"mix":976},{"t":2143.468,"mix":977},{"t":2144.001,
"mix":978},{"t":2144.505,"mix":979},{"t":2145.032,"mix":980},{"t":2145.579,"mix":981},{"t":2146.112,
"mix":982},{"t":2146.643,"mix":983},{"t":2147.201,"mix":984},{"t":2147.743,"mix":985},{"t":2148.247,
"mix":986},{"t":2148.79,"mix":987},{"t":2149.321,"mix":988},{"t":2149.901,"mix":989},{"t":2150.423,
"mix":990},{"t":2150.929,"mix":991},{"t":2151.507,"mix":992},{"t":2152.045,"mix":993},{"t":2152.567,
"mix":994},{"t":2153.147,"mix":995},{"t":2153.667,"mix":996},{"t":2154.249,"mix":997},{"t":2154.772,
"mix":998},{"t":2155.303,"mix":999},{"t":2155.79,"mix":1000},{"t":2156.302,"mix":1001},{"t":2156.803,
"mix":1002},{"t":2157.312,"mix":1003},{"t":2157.852,"mix":1004},{"t":2158.407,"mix":1005},
{"t":2158.983,"mix":1006},{"t":2159.463,"mix":1007},{"t":2160.032,"mix":1008},{"t":2160.55,
"mix":1009},{"t":2161.093,"mix":1010},{"t":2161.575,"mix":1011},{"t":2162.143,"mix":1012},
{"t":2162.68,"mix":1013},{"t":2163.189,"mix":1014},{"t":2163.743,"mix":1015},{"t":2164.302,
"mix":1016},{"t":2164.868,"mix":1017},{"t":2165.374,"mix":1018},{"t":2165.937,"mix":1019},
{"t":2166.474,"mix":1020},{"t":2167.017,"mix":1021},{"t":2167.501,"mix":1022},{"t":2168.033,
"mix":1023},{"t":2168.641,"mix":1024},{"t":2169.15,"mix":1025},{"t":2169.628,"mix":1026},
{"t":2170.147,"mix":1027},{"t":2170.673,"mix":1028},{"t":2171.205,"mix":1029},{"t":2171.71,
"mix":1030},{"t":2172.256,"mix":1031},{"t":2172.874,"mix":1032},{"t":2173.405,"mix":1033},
{"t":2173.849,"mix":1034},{"t":2174.393,"mix":1035},{"t":2174.937,"mix":1036},{"t":2175.432,
"mix":1037},{"t":2175.93,"mix":1038},{"t":2176.463,"mix":1039},{"t":2177.03,"mix":1040},{"t":2177.602,
"mix":1041},{"t":2178.104,"mix":1042},{"t":2178.597,"mix":1043},{"t":2179.081,"mix":1044},
{"t":2179.683,"mix":1045},{"t":2180.216,"mix":1046},{"t":2180.686,"mix":1047},{"t":2181.24,
"mix":1048},{"t":2181.765,"mix":1049},{"t":2182.308,"mix":1050},{"t":2182.823,"mix":1051},
{"t":2183.344,"mix":1052},{"t":2183.886,"mix":1053},{"t":2184.373,"mix":1054},{"t":2184.867,
"mix":1055},{"t":2185.373,"mix":1056},{"t":2185.919,"mix":1057},{"t":2186.432,"mix":1058},
{"t":2186.951,"mix":1059},{"t":2187.457,"mix":1060},{"t":2187.979,"mix":1061},{"t":2188.456,
"mix":1062},{"t":2189.03,"mix":1063},{"t":2189.528,"mix":1064},{"t":2190.044,"mix":1065},
{"t":2190.529,"mix":1066},{"t":2191.049,"mix":1067},{"t":2191.579,"mix":1068},{"t":2192.106,
"mix":1069},{"t":2192.559,"mix":1070},{"t":2193.114,"mix":1071},{"t":2193.646,"mix":1072},
{"t":2194.191,"mix":1073},{"t":2194.649,"mix":1074},{"t":2195.215,"mix":1075},{"t":2195.723,
"mix":1076},{"t":2196.271,"mix":1077},{"t":2196.864,"mix":1078},{"t":2197.378,"mix":1079},
{"t":2197.916,"mix":1080},{"t":2198.476,"mix":1081},{"t":2198.993,"mix":1082},{"t":2199.529,
"mix":1083},{"t":2200.103,"mix":1084},{"t":2200.576,"mix":1085},{"t":2201.126,"mix":1086},
{"t":2201.662,"mix":1087},{"t":2202.22,"mix":1088},{"t":2202.729,"mix":1089},{"t":2203.251,
"mix":1090},{"t":2203.833,"mix":1091},{"t":2204.411,"mix":1092},{"t":2204.89,"mix":1093},
{"t":2205.437,"mix":1094},{"t":2205.988,"mix":1095},{"t":2206.538,"mix":1096},{"t":2207.025,
"mix":1097},{"t":2207.529,"mix":1098},{"t":2208.077,"mix":1099},{"t":2208.544,"mix":1100},
{"t":2209.074,"mix":1101},{"t":2209.594,"mix":1102},{"t":2210.103,"mix":1103},{"t":2210.683,
"mix":1104},{"t":2211.197,"mix":1105},{"t":2211.716,"mix":1110},{"t":2212.248,"mix":1111},
{"t":2212.774,"mix":1112},{"t":2213.354,"mix":1113},{"t":2213.79,"mix":1114},{"t":2214.035,
"mix":1115},{"t":2214.707,"mix":1116},{"t":2215.433,"mix":1117},{"t":2215.965,"mix":1118},
{"t":2216.656,"mix":1119},{"t":2217.164,"mix":1120},{"t":2217.701,"mix":1121},{"t":2218.269,
"mix":1122},{"t":2218.872,"mix":1123},{"t":2219.423,"mix":1124},{"t":2219.965,"mix":1125},
{"t":2220.527,"mix":1126},{"t":2221.093,"mix":1127},{"t":2221.681,"mix":1128},{"t":2222.166,
"mix":1129},{"t":2222.741,"mix":1130},{"t":2223.321,"mix":1131},{"t":2224.011,"mix":1132},
{"t":2224.642,"mix":1133},{"t":2225.135,"mix":1134},{"t":2225.633,"mix":1135},{"t":2226.269,
"mix":1136},{"t":2226.859,"mix":1137},{"t":2227.441,"mix":1138},{"t":2228.086,"mix":1139},
{"t":2228.608,"mix":1140},{"t":2229.182,"mix":1141},{"t":2229.807,"mix":1142},{"t":2230.398,
"mix":1143},{"t":2230.931,"mix":1144},{"t":2231.401,"mix":1145},{"t":2232.012,"mix":1146},
{"t":2232.575,"mix":1147},{"t":2233.166,"mix":1148},{"t":2233.69,"mix":1149},{"t":2234.275,
"mix":1150},{"t":2234.841,"mix":1151},{"t":2235.4,"mix":1152},{"t":2235.924,"mix":1153},{"t":2236.506,
"mix":1154},{"t":2237.1,"mix":1155},{"t":2237.659,"mix":1156},{"t":2238.175,"mix":1157},{"t":2238.739,
"mix":1158},{"t":2239.309,"mix":1159},{"t":2239.821,"mix":1160},{"t":2240.382,"mix":1161},
{"t":2240.931,"mix":1162},{"t":2241.472,"mix":1163},{"t":2242.025,"mix":1164},{"t":2242.609,
"mix":1165},{"t":2243.145,"mix":1166},{"t":2243.746,"mix":1167},{"t":2244.3,"mix":1168},{"t":2244.845,
"mix":1169},{"t":2245.418,"mix":1170},{"t":2246.006,"mix":1171},{"t":2246.627,"mix":1172},
{"t":2247.267,"mix":1173},{"t":2247.918,"mix":1174},{"t":2248.429,"mix":1175},{"t":2249.049,
"mix":1176},{"t":2249.606,"mix":1177},{"t":2250.187,"mix":1178},{"t":2250.75,"mix":1179},
{"t":2251.365,"mix":1180},{"t":2251.954,"mix":1181},{"t":2252.506,"mix":1182},{"t":2253.102,
"mix":1183},{"t":2253.661,"mix":1184},{"t":2254.301,"mix":1185},{"t":2254.902,"mix":1186},
{"t":2255.512,"mix":1187},{"t":2256.115,"mix":1188},{"t":2256.835,"mix":1189},{"t":2257.445,
"mix":1190},{"t":2258.058,"mix":1191},{"t":2258.716,"mix":1192},{"t":2259.498,"mix":1193},
{"t":2260.059,"mix":1194},{"t":2260.694,"mix":1195},{"t":2261.438,"mix":1196},{"t":2262.048,
"mix":1197},{"t":2262.762,"mix":1198},{"t":2263.325,"mix":1199},{"t":2264.037,"mix":1200},
{"t":2264.762,"mix":1201},{"t":2265.424,"mix":1202},{"t":2266.062,"mix":1203},{"t":2266.751,
"mix":1204},{"t":2267.334,"mix":1205},{"t":2267.879,"mix":1206},{"t":2268.429,"mix":1207},
{"t":2269.01,"mix":1208},{"t":2269.544,"mix":1147},{"t":2270.157,"mix":1148},{"t":2270.682,
"mix":1149},{"t":2271.265,"mix":1150},{"t":2271.812,"mix":1151},{"t":2272.402,"mix":1152},
{"t":2272.922,"mix":1153},{"t":2273.471,"mix":1154},{"t":2274.078,"mix":1155},{"t":2274.668,
"mix":1156},{"t":2275.233,"mix":1157},{"t":2275.795,"mix":1158},{"t":2276.38,"mix":1159},
{"t":2276.853,"mix":1160},{"t":2277.439,"mix":1161},{"t":2277.963,"mix":1162},{"t":2278.59,
"mix":1163},{"t":2279.169,"mix":1164},{"t":2279.741,"mix":1165},{"t":2280.309,"mix":1166},
{"t":2280.865,"mix":1167},{"t":2281.426,"mix":1168},{"t":2281.997,"mix":1169},{"t":2282.552,
"mix":1170},{"t":2283.139,"mix":1171},{"t":2283.718,"mix":1172},{"t":2284.476,"mix":1173},
{"t":2285.15,"mix":1174},{"t":2285.732,"mix":1175},{"t":2286.321,"mix":1176},{"t":2286.888,
"mix":1177},{"t":2287.425,"mix":1178},{"t":2288.016,"mix":1179},{"t":2288.614,"mix":1180},
{"t":2289.247,"mix":1181},{"t":2289.809,"mix":1182},{"t":2290.39,"mix":1183},{"t":2290.976,
"mix":1184},{"t":2291.622,"mix":1185},{"t":2292.177,"mix":1186},{"t":2292.752,"mix":1187},
{"t":2293.407,"mix":1188},{"t":2294.1,"mix":1189},{"t":2294.742,"mix":1190},{"t":2295.378,
"mix":1191},{"t":2296.171,"mix":1192},{"t":2296.857,"mix":1193},{"t":2297.451,"mix":1194},
{"t":2298.031,"mix":1195},{"t":2298.742,"mix":1196},{"t":2299.342,"mix":1197},{"t":2300.089,
"mix":1198},{"t":2300.735,"mix":1199},{"t":2301.436,"mix":1200},{"t":2302.11,"mix":1201},
{"t":2302.756,"mix":1202},{"t":2303.366,"mix":1209},{"t":2304.034,"mix":1210},{"t":2304.572,
"mix":1211},{"t":2305.126,"mix":1212},{"t":2305.615,"mix":1213},{"t":2306.176,"mix":1214},
{"t":2306.741,"mix":1215},{"t":2307.319,"mix":1216},{"t":2307.857,"mix":1217},{"t":2308.406,
"mix":1218},{"t":2308.924,"mix":1219},{"t":2309.446,"mix":1220},{"t":2310.059,"mix":1221},
{"t":2310.556,"mix":1222},{"t":2311.084,"mix":1223},{"t":2311.571,"mix":1224},{"t":2312.055,
"mix":1225},{"t":2312.516,"mix":1226},{"t":2313.079,"mix":1227},{"t":2313.638,"mix":1228},
{"t":2314.261,"mix":1229},{"t":2314.784,"mix":1230},{"t":2315.308,"mix":1231},{"t":2315.855,
"mix":1232},{"t":2316.381,"mix":1233},{"t":2316.874,"mix":1234},{"t":2317.408,"mix":1235},
{"t":2317.948,"mix":1236},{"t":2318.43,"mix":1237},{"t":2319.023,"mix":1238},{"t":2319.551,
"mix":1239},{"t":2320.066,"mix":1240},{"t":2320.621,"mix":1241},{"t":2321.202,"mix":1242},
{"t":2321.805,"mix":1243},{"t":2322.374,"mix":1244},{"t":2322.882,"mix":1245},{"t":2323.395,
"mix":1246},{"t":2323.966,"mix":1247},{"t":2324.516,"mix":1248},{"t":2325.025,"mix":1249},
{"t":2325.528,"mix":1250},{"t":2326.026,"mix":1251},{"t":2326.624,"mix":1252},{"t":2327.156,
"mix":1253},{"t":2327.715,"mix":1254},{"t":2328.224,"mix":1255},{"t":2328.784,"mix":1256},
{"t":2329.314,"mix":1257},{"t":2329.866,"mix":1258},{"t":2330.402,"mix":1259},{"t":2330.935,
"mix":1260},{"t":2331.432,"mix":1261},{"t":2332.002,"mix":1262},{"t":2332.535,"mix":1263},
{"t":2333.057,"mix":1264},{"t":2333.601,"mix":1265},{"t":2334.159,"mix":1266},{"t":2334.67,
"mix":1267},{"t":2335.213,"mix":1268},{"t":2335.723,"mix":1269},{"t":2336.264,"mix":1270},
{"t":2336.769,"mix":1271},{"t":2337.333,"mix":1272},{"t":2337.841,"mix":1273},{"t":2338.334,
"mix":1274},{"t":2338.896,"mix":1275},{"t":2339.408,"mix":1276},{"t":2339.952,"mix":1277},
{"t":2340.508,"mix":1278},{"t":2341.044,"mix":1279},{"t":2341.543,"mix":1280},{"t":2342.085,
"mix":1281},{"t":2342.64,"mix":1282},{"t":2343.178,"mix":1283},{"t":2343.71,"mix":1284},{"t":2344.303,
"mix":1285},{"t":2344.812,"mix":1286},{"t":2345.339,"mix":1287},{"t":2345.87,"mix":1288},
{"t":2346.375,"mix":1289},{"t":2346.958,"mix":1290},{"t":2347.499,"mix":1291},{"t":2348.015,
"mix":1292},{"t":2348.554,"mix":1293},{"t":2349.058,"mix":1294},{"t":2349.54,"mix":1295},
{"t":2350.077,"mix":1296},{"t":2350.592,"mix":1297},{"t":2351.155,"mix":1298},{"t":2351.656,
"mix":1299},{"t":2352.145,"mix":1300},{"t":2352.748,"mix":1301},{"t":2353.261,"mix":1302},
{"t":2353.748,"mix":1303},{"t":2354.278,"mix":1304},{"t":2354.809,"mix":1305},{"t":2355.345,
"mix":1306},{"t":2355.818,"mix":1307},{"t":2356.284,"mix":1308},{"t":2356.928,"mix":1309},
{"t":2357.5,"mix":1310},{"t":2357.973,"mix":1311},{"t":2358.5,"mix":1312},{"t":2359.006,"mix":1313},
{"t":2359.573,"mix":1314},{"t":2360.109,"mix":1315},{"t":2360.624,"mix":1316},{"t":2361.106,
"mix":1317},{"t":2361.674,"mix":1318},{"t":2362.167,"mix":1319},{"t":2362.755,"mix":1320},
{"t":2363.266,"mix":1321},{"t":2363.785,"mix":1322},{"t":2364.255,"mix":1323},{"t":2364.759,
"mix":1324},{"t":2365.26,"mix":1325},{"t":2365.764,"mix":1326},{"t":2366.296,"mix":1327},
{"t":2366.815,"mix":1328},{"t":2367.352,"mix":1329},{"t":2367.858,"mix":1330},{"t":2368.364,
"mix":1331},{"t":2368.904,"mix":1332},{"t":2369.424,"mix":1333},{"t":2369.96,"mix":1334},
{"t":2370.441,"mix":1335},{"t":2371.037,"mix":1336},{"t":2371.562,"mix":1337},{"t":2372.102,
"mix":1338},{"t":2372.631,"mix":1339},{"t":2373.224,"mix":1340},{"t":2373.712,"mix":1341},
{"t":2374.263,"mix":1342},{"t":2374.791,"mix":1343},{"t":2375.317,"mix":1344},{"t":2375.863,
"mix":1345},{"t":2376.417,"mix":1346},{"t":2376.946,"mix":1347},{"t":2377.51,"mix":1348},
{"t":2378.048,"mix":1349},{"t":2378.609,"mix":1350},{"t":2379.123,"mix":1351},{"t":2379.658,
"mix":1352},{"t":2380.229,"mix":1353},{"t":2380.746,"mix":1354},{"t":2381.285,"mix":1355},
{"t":2381.825,"mix":1356},{"t":2382.322,"mix":1357},{"t":2382.89,"mix":1358},{"t":2383.474,
"mix":1359},{"t":2384.012,"mix":1360},{"t":2384.545,"mix":1361},{"t":2385.074,"mix":1362},
{"t":2385.611,"mix":1363},{"t":2386.174,"mix":1364},{"t":2386.693,"mix":1365},{"t":2387.249,
"mix":1366},{"t":2387.744,"mix":1367},{"t":2388.288,"mix":1368},{"t":2388.761,"mix":1369},
{"t":2389.324,"mix":1370},{"t":2389.845,"mix":1371},{"t":2390.382,"mix":1372},{"t":2390.933,
"mix":1373},{"t":2391.417,"mix":1374},{"t":2391.965,"mix":1375},{"t":2392.503,"mix":1376},
{"t":2392.987,"mix":1377},{"t":2393.563,"mix":1378},{"t":2394.095,"mix":1379},{"t":2394.639,
"mix":1380},{"t":2395.153,"mix":1381},{"t":2395.732,"mix":1382},{"t":2396.257,"mix":1383},
{"t":2396.831,"mix":1384},{"t":2397.318,"mix":1385},{"t":2397.857,"mix":1386},{"t":2398.417,
"mix":1387},{"t":2398.936,"mix":1388},{"t":2399.426,"mix":1389},{"t":2399.974,"mix":1390},
{"t":2400.51,"mix":1391},{"t":2401.034,"mix":1392},{"t":2401.586,"mix":1393},{"t":2402.137,
"mix":1394},{"t":2402.833,"mix":1395},{"t":2403.608,"mix":1396},{"t":2417.151,"mix":1397},
{"t":2418.529,"mix":1398},{"t":2419.387,"mix":1399},{"t":2420.178,"mix":1400},{"t":2421.055,
"mix":1401},{"t":2421.988,"mix":1402},{"t":2422.853,"mix":1403},{"t":2423.774,"mix":1404},
{"t":2424.73,"mix":1405},{"t":2425.647,"mix":1406},{"t":2427.043,"mix":1407},{"t":2432.778,
"mix":1408},{"t":2434.083,"mix":1409},{"t":2435.122,"mix":1410},{"t":2436.073,"mix":1411},
{"t":2437.069,"mix":1412},{"t":2438.008,"mix":1413},{"t":2439.009,"mix":1414},{"t":2439.95,
"mix":1415},{"t":2440.909,"mix":1416},{"t":2442.004,"mix":1417},{"t":2442.952,"mix":1418},
{"t":2443.899,"mix":1419},{"t":2444.821,"mix":1420},{"t":2445.788,"mix":1421},{"t":2446.706,
"mix":1422},{"t":2447.618,"mix":1423},{"t":2448.504,"mix":1424},{"t":2449.452,"mix":1425},
{"t":2450.513,"mix":1426},{"t":2451.822,"mix":1427},{"t":2454.481,"mix":1428},{"t":2455.332,
"mix":1429},{"t":2456.239,"mix":1430},{"t":2457.207,"mix":1431},{"t":2458.107,"mix":1432},
{"t":2459.04,"mix":1433},{"t":2460.094,"mix":1434},{"t":2461.621,"mix":1435},{"t":2464.173,
"mix":1436},{"t":2465.014,"mix":1437},{"t":2465.914,"mix":1438},{"t":2466.846,"mix":1439},
{"t":2467.771,"mix":1440},{"t":2468.78,"mix":1441},{"t":2469.763,"mix":1442},{"t":2470.671,
"mix":1443},{"t":2471.65,"mix":1444},{"t":2472.64,"mix":1445},{"t":2473.564,"mix":1446},{"t":2474.435,
"mix":1447},{"t":2475.343,"mix":1448},{"t":2476.267,"mix":1441},{"t":2477.269,"mix":1442},
{"t":2478.198,"mix":1443},{"t":2479.182,"mix":1444},{"t":2480.125,"mix":1445},{"t":2481.053,
"mix":1446},{"t":2481.985,"mix":1449},{"t":2482.641,"mix":1450},{"t":2482.957,"mix":1451},
{"t":2483.828,"mix":1452},{"t":2484.774,"mix":1453},{"t":2485.968,"mix":1454},{"t":2489.41,
"mix":1455},{"t":2490.322,"mix":1456},{"t":2491.248,"mix":1457},{"t":2492.176,"mix":1458},
{"t":2492.89,"mix":1450},{"t":2493.127,"mix":1451},{"t":2493.987,"mix":1452},{"t":2494.999,
"mix":1453},{"t":2496.105,"mix":1454},{"t":2499.851,"mix":1455},{"t":2500.765,"mix":1456},
{"t":2501.685,"mix":1457},{"t":2502.611,"mix":1458},{"t":2503.33,"mix":1459},{"t":2503.627,
"mix":1460},{"t":2504.529,"mix":1461},{"t":2505.489,"mix":1462},{"t":2506.418,"mix":1463},
{"t":2507.36,"mix":1464},{"t":2508.388,"mix":1465},{"t":2509.334,"mix":1466},{"t":2510.349,
"mix":1467},{"t":2511.038,"mix":1459},{"t":2511.326,"mix":1460},{"t":2512.265,"mix":1461},
{"t":2513.223,"mix":1462},{"t":2514.192,"mix":1463},{"t":2515.102,"mix":1464},{"t":2516.095,
"mix":1465},{"t":2517.083,"mix":1466},{"t":2518.069,"mix":1467},{"t":2518.806,"mix":1468},
{"t":2519.092,"mix":1469},{"t":2519.982,"mix":1470},{"t":2521.042,"mix":1471},{"t":2522.421,
"mix":1472},{"t":2524.919,"mix":1473},{"t":2525.902,"mix":1474},{"t":2526.882,"mix":1475},
{"t":2527.842,"mix":1476},{"t":2528.655,"mix":1468},{"t":2528.964,"mix":1469},{"t":2529.889,
"mix":1470},{"t":2530.913,"mix":1471},{"t":2532.661,"mix":1472},{"t":2534.796,"mix":1473},
{"t":2535.862,"mix":1474},{"t":2536.823,"mix":1475},{"t":2537.863,"mix":1476},{"t":2538.648,
"mix":1477},{"t":2538.96,"mix":1478},{"t":2540.084,"mix":1479},{"t":2541.085,"mix":1480},
{"t":2542.095,"mix":1481},{"t":2543.059,"mix":1482},{"t":2544.117,"mix":1483},{"t":2545.064,
"mix":1484},{"t":2546.009,"mix":1485},{"t":2547.051,"mix":1486},{"t":2548.036,"mix":1487},
{"t":2548.998,"mix":1488},{"t":2549.988,"mix":1489},{"t":2550.939,"mix":1490},{"t":2551.962,
"mix":1491},{"t":2552.913,"mix":1492},{"t":2553.895,"mix":1493},{"t":2554.924,"mix":1494},
{"t":2555.818,"mix":1495},{"t":2556.981,"mix":1496},{"t":2558.334,"mix":1497},{"t":2561.233,
"mix":1498},{"t":2562.162,"mix":1499},{"t":2563.166,"mix":1500},{"t":2564.194,"mix":1501},
{"t":2565.134,"mix":1502},{"t":2566.119,"mix":1503},{"t":2567.137,"mix":1504},{"t":2568.609,
"mix":1505},{"t":2571.944,"mix":1506},{"t":2572.873,"mix":1507},{"t":2573.827,"mix":1508},
{"t":2574.828,"mix":1509},{"t":2575.787,"mix":1510},{"t":2576.835,"mix":1511},{"t":2577.701,
"mix":1512},{"t":2578.832,"mix":1513},{"t":2579.653,"mix":1514},{"t":2580.568,"mix":1515},
{"t":2581.528,"mix":1516},{"t":2582.413,"mix":1517},{"t":2583.449,"mix":1518},{"t":2584.511,
"mix":1519},{"t":2585.5,"mix":1520},{"t":2586.4,"mix":1521},{"t":2587.361,"mix":1522},{"t":2588.335,
"mix":1523},{"t":2589.222,"mix":1524},{"t":2590.182,"mix":1525},{"t":2591.127,"mix":1526},
{"t":2592.102,"mix":1527},{"t":2593.019,"mix":1528},{"t":2594.008,"mix":1529},{"t":2594.956,
"mix":1530},{"t":2595.904,"mix":1531},{"t":2596.775,"mix":1532},{"t":2597.758,"mix":1533},
{"t":2598.667,"mix":1534},{"t":2599.673,"mix":1535},{"t":2600.594,"mix":1536},{"t":2601.58,
"mix":1537},{"t":2602.457,"mix":1538},{"t":2603.425,"mix":1539},{"t":2604.322,"mix":1540},
{"t":2605.276,"mix":1541},{"t":2606.206,"mix":1542},{"t":2607.183,"mix":1543},{"t":2608.102,
"mix":1544},{"t":2609.053,"mix":1545},{"t":2609.948,"mix":1546},{"t":2610.853,"mix":1547},
{"t":2611.805,"mix":1548},{"t":2612.793,"mix":1549},{"t":2613.677,"mix":1550},{"t":2614.628,
"mix":1551},{"t":2615.608,"mix":1552},{"t":2616.508,"mix":1553},{"t":2617.5,"mix":1554},{"t":2618.367,
"mix":1555},{"t":2619.328,"mix":1556},{"t":2620.247,"mix":1557},{"t":2621.145,"mix":1558},
{"t":2622.087,"mix":1559},{"t":2623.018,"mix":1560},{"t":2623.937,"mix":1561},{"t":2624.917,
"mix":1562},{"t":2625.913,"mix":1563},{"t":2626.86,"mix":1564},{"t":2627.777,"mix":1565},
{"t":2628.755,"mix":1566},{"t":2629.691,"mix":1567},{"t":2630.619,"mix":1568},{"t":2631.534,
"mix":1569},{"t":2632.508,"mix":1570},{"t":2633.407,"mix":1571},{"t":2634.354,"mix":1572},
{"t":2635.28,"mix":1573},{"t":2636.255,"mix":1574},{"t":2637.185,"mix":1575},{"t":2638.124,
"mix":1576},{"t":2639.107,"mix":1577},{"t":2640.128,"mix":1578},{"t":2641.123,"mix":1579},
{"t":2642.094,"mix":1580},{"t":2643.062,"mix":1581},{"t":2644.012,"mix":1582},{"t":2644.916,
"mix":1583},{"t":2645.885,"mix":1584},{"t":2646.935,"mix":1585},{"t":2647.847,"mix":1586},
{"t":2648.793,"mix":1587},{"t":2649.713,"mix":1588},{"t":2650.629,"mix":1589},{"t":2651.604,
"mix":1590},{"t":2652.545,"mix":1591},{"t":2653.527,"mix":1592},{"t":2654.459,"mix":1593},
{"t":2655.438,"mix":1594},{"t":2656.403,"mix":1595},{"t":2657.357,"mix":1596},{"t":2658.352,
"mix":1597},{"t":2659.297,"mix":1598},{"t":2660.264,"mix":1599},{"t":2661.244,"mix":1600},
{"t":2662.193,"mix":1601},{"t":2663.178,"mix":1602},{"t":2664.176,"mix":1603},{"t":2665.121,
"mix":1604},{"t":2666.076,"mix":1605},{"t":2667.008,"mix":1606},{"t":2667.925,"mix":1607},
{"t":2668.885,"mix":1608},{"t":2669.864,"mix":1609},{"t":2670.85,"mix":1610},{"t":2671.738,
"mix":1611},{"t":2672.698,"mix":1612},{"t":2673.633,"mix":1613},{"t":2674.614,"mix":1614},
{"t":2675.566,"mix":1615},{"t":2676.483,"mix":1616},{"t":2677.447,"mix":1617},{"t":2678.414,
"mix":1618},{"t":2679.34,"mix":1619},{"t":2680.267,"mix":1620},{"t":2681.221,"mix":1621},
{"t":2682.189,"mix":1622},{"t":2683.14,"mix":1623},{"t":2684.086,"mix":1624},{"t":2685.012,
"mix":1625},{"t":2685.993,"mix":1626},{"t":2686.913,"mix":1627},{"t":2687.863,"mix":1628},
{"t":2688.847,"mix":1629},{"t":2689.88,"mix":1630},{"t":2690.867,"mix":1631},{"t":2691.829,
"mix":1632},{"t":2692.741,"mix":1633},{"t":2693.772,"mix":1634},{"t":2694.687,"mix":1635},
{"t":2695.613,"mix":1636},{"t":2696.605,"mix":1637},{"t":2697.584,"mix":1638},{"t":2698.56,
"mix":1639},{"t":2699.514,"mix":1640},{"t":2700.429,"mix":1641},{"t":2701.47,"mix":1642},
{"t":2702.411,"mix":1643},{"t":2703.3,"mix":1644},{"t":2704.243,"mix":1645},{"t":2705.175,
"mix":1646},{"t":2706.094,"mix":1647},{"t":2707.008,"mix":1648},{"t":2707.943,"mix":1649},
{"t":2708.925,"mix":1650},{"t":2709.79,"mix":1651},{"t":2710.631,"mix":1652},{"t":2711.537,
"mix":1653},{"t":2712.411,"mix":1654},{"t":2713.291,"mix":1655},{"t":2714.156,"mix":1656},
{"t":2715.158,"mix":1657},{"t":2716.331,"mix":1658},{"t":2717.814,"mix":1659},{"t":2719.468,
"mix":1660},{"t":2720.714,"mix":1661},{"t":2721.7,"mix":1662},{"t":2722.646,"mix":1663},{"t":2723.731,
"mix":1664},{"t":2724.689,"mix":1665},{"t":2725.646,"mix":1666},{"t":2726.626,"mix":1667},
{"t":2727.79,"mix":1668},{"t":2728.802,"mix":1669},{"t":2729.89,"mix":1670},{"t":2730.871,
"mix":1671},{"t":2731.888,"mix":1672},{"t":2732.98,"mix":1673},{"t":2733.955,"mix":1674},
{"t":2734.996,"mix":1675},{"t":2735.956,"mix":1676},{"t":2737.084,"mix":1677},{"t":2738.042,
"mix":1678},{"t":2739.161,"mix":1679},{"t":2740.096,"mix":1680},{"t":2741.048,"mix":1681},
{"t":2741.982,"mix":1682},{"t":2743.045,"mix":1683},{"t":2743.966,"mix":1684},{"t":2744.968,
"mix":1685},{"t":2745.903,"mix":1686},{"t":2746.858,"mix":1687},{"t":2747.892,"mix":1688},
{"t":2748.829,"mix":1689},{"t":2749.858,"mix":1690},{"t":2750.843,"mix":1691},{"t":2751.8,
"mix":1692},{"t":2752.811,"mix":1693},{"t":2753.661,"mix":1694},{"t":2754.697,"mix":1695},
{"t":2755.645,"mix":1696},{"t":2756.639,"mix":1697},{"t":2757.596,"mix":1698},{"t":2758.561,
"mix":1699},{"t":2759.538,"mix":1700},{"t":2760.467,"mix":1701},{"t":2761.43,"mix":1702},
{"t":2762.408,"mix":1703},{"t":2763.377,"mix":1704},{"t":2764.373,"mix":1705},{"t":2765.397,
"mix":1706},{"t":2766.337,"mix":1707},{"t":2767.323,"mix":1708},{"t":2768.288,"mix":1709},
{"t":2769.273,"mix":1710},{"t":2770.169,"mix":1711},{"t":2771.2,"mix":1712},{"t":2772.174,
"mix":1713},{"t":2773.115,"mix":1714},{"t":2774.118,"mix":1715},{"t":2775.084,"mix":1716},
{"t":2776.04,"mix":1717},{"t":2777.005,"mix":1718},{"t":2777.998,"mix":1719},{"t":2779.021,
"mix":1720},{"t":2779.939,"mix":1721},{"t":2780.875,"mix":1722},{"t":2781.865,"mix":1723},
{"t":2782.821,"mix":1724},{"t":2783.766,"mix":1725},{"t":2784.728,"mix":1726},{"t":2785.628,
"mix":1727},{"t":2786.573,"mix":1728},{"t":2787.498,"mix":1729},{"t":2788.471,"mix":1730},
{"t":2789.447,"mix":1731},{"t":2790.452,"mix":1732},{"t":2791.409,"mix":1733},{"t":2792.358,
"mix":1734},{"t":2793.305,"mix":1735},{"t":2794.19,"mix":1736},{"t":2795.169,"mix":1737},
{"t":2796.109,"mix":1738},{"t":2797.044,"mix":1739},{"t":2797.993,"mix":1740},{"t":2798.889,
"mix":1741},{"t":2799.815,"mix":1742},{"t":2800.72,"mix":1743},{"t":2801.636,"mix":1744},
{"t":2802.66,"mix":1745},{"t":2803.562,"mix":1746},{"t":2804.698,"mix":1747},{"t":2805.87,
"mix":1748},{"t":2807.143,"mix":1749},{"t":2808.872,"mix":1750},{"t":2816.404,"mix":1751},
{"t":2819.965,"mix":1752},{"t":2823.489,"mix":1753},{"t":2827.644,"mix":1754},{"t":2831.131,
"mix":1755},{"t":2834.757,"mix":1756},{"t":2838.28,"mix":1757},{"t":2841.488,"mix":1758},
{"t":2845.854,"mix":1759},{"t":2850.095,"mix":1760},{"t":2853.59,"mix":1761},{"t":2857.948,
"mix":1762},{"t":2861.679,"mix":1763},{"t":2865.127,"mix":1764},{"t":2868.969,"mix":1765},
{"t":2872.24,"mix":1766},{"t":2876.141,"mix":1767},{"t":2879.474,"mix":1768},{"t":2882.453,
"mix":1769},{"t":2885.748,"mix":1770},{"t":2888.712,"mix":1771},{"t":2892.102,"mix":1772},
{"t":2895.101,"mix":1773},{"t":2898.219,"mix":1774},{"t":2901.36,"mix":1775},{"t":2904.371,
"mix":1776},{"t":2907.486,"mix":1777},{"t":2910.491,"mix":1778},{"t":2913.542,"mix":1779},
{"t":2916.566,"mix":1780},{"t":2919.613,"mix":1781},{"t":2922.698,"mix":1782},{"t":2926.073,
"mix":1783},{"t":2929.215,"mix":1784},{"t":2932.365,"mix":1785},{"t":2935.476,"mix":1786},
{"t":2938.743,"mix":1787},{"t":2941.881,"mix":1788},{"t":2944.944,"mix":1789},{"t":2948.139,
"mix":1790},{"t":2951.337,"mix":1791},{"t":2954.386,"mix":1792},{"t":2957.554,"mix":1793},
{"t":2960.657,"mix":1794},{"t":2963.906,"mix":1795},{"t":2966.979,"mix":1796},{"t":2970.026,
"mix":1797},{"t":2973.253,"mix":1798},{"t":2976.46,"mix":1799},{"t":2979.572,"mix":1800},
{"t":2982.703,"mix":1801},{"t":2985.821,"mix":1802},{"t":2988.988,"mix":1803},{"t":2991.982,
"mix":1804},{"t":2995.085,"mix":1805},{"t":2998.124,"mix":1806},{"t":3001.597,"mix":1807},
{"t":3004.719,"mix":1808},{"t":3007.76,"mix":1809},{"t":3010.819,"mix":1810},{"t":3014,"mix":1811},
{"t":3017.005,"mix":1812},{"t":3020.344,"mix":1813},{"t":3023.371,"mix":1814},{"t":3026.739,
"mix":1815},{"t":3030.025,"mix":1816},{"t":3033.224,"mix":1817},{"t":3036.584,"mix":1818},
{"t":3039.805,"mix":1819},{"t":3043.179,"mix":1820},{"t":3046.532,"mix":1821},{"t":3049.784,
"mix":1822},{"t":3053.092,"mix":1823},{"t":3056.298,"mix":1824},{"t":3059.529,"mix":1825},
{"t":3062.79,"mix":1826},{"t":3065.961,"mix":1827},{"t":3069.243,"mix":1828},{"t":3072.443,
"mix":1829},{"t":3075.654,"mix":1830},{"t":3078.794,"mix":1831},{"t":3082.093,"mix":1832},
{"t":3085.465,"mix":1833},{"t":3086.817,"mix":1834},{"t":3088.006,"mix":1835},{"t":3089.214,
"mix":1836},{"t":3090.452,"mix":1837},{"t":3091.703,"mix":1838},{"t":3092.951,"mix":1839},
{"t":3094.158,"mix":1840},{"t":3095.409,"mix":1841},{"t":3096.63,"mix":1842},{"t":3097.894,
"mix":1843},{"t":3099.208,"mix":1844},{"t":3100.506,"mix":1845},{"t":3101.757,"mix":1846},
{"t":3103.005,"mix":1847},{"t":3104.279,"mix":1848},{"t":3105.56,"mix":1849},{"t":3106.891,
"mix":1850},{"t":3108.148,"mix":1851},{"t":3109.367,"mix":1852},{"t":3110.623,"mix":1853},
{"t":3111.972,"mix":1854},{"t":3113.148,"mix":1855},{"t":3114.429,"mix":1856},{"t":3115.705,
"mix":1857},{"t":3116.969,"mix":1858},{"t":3118.213,"mix":1859},{"t":3119.568,"mix":1860},
{"t":3120.862,"mix":1861},{"t":3122.085,"mix":1862},{"t":3123.322,"mix":1863},{"t":3124.685,
"mix":1864},{"t":3126.055,"mix":1865},{"t":3127.441,"mix":1866},{"t":3128.724,"mix":1867},
{"t":3130.067,"mix":1868},{"t":3131.478,"mix":1869},{"t":3132.786,"mix":1870},{"t":3134.088,
"mix":1871},{"t":3135.46,"mix":1872},{"t":3136.683,"mix":1873},{"t":3138.13,"mix":1874},{"t":3139.723,
"mix":1875},{"t":3158.140,"mix":1876}]
'''  # Use the full JSON
timestamps1 = json.loads(timestamps_json)

def parse_start_time_from_url(youtube_url):
    parsed_url = urlparse.urlparse(youtube_url)
    query_params = urlparse.parse_qs(parsed_url.query)
    start_time = query_params.get('t', ['0'])[0]  # Default to '0' if not provided
    if 's' in start_time:
        # Extract time in seconds from the URL parameter
        time_seconds = int(re.search(r'\d+', start_time).group())
        return time_seconds
    else:
        return int(start_time)

def download_audio_with_retries(youtube_url, output_path, max_retries=5):
    retry_count = 0
    while retry_count < max_retries:
        try:
            yt = YouTube(youtube_url)
            audio_stream = yt.streams.filter(only_audio=True).first()
            if not audio_stream:
                print("No audio stream found.")
                return None
            # Download the audio stream directly without conversion
            downloaded_file = audio_stream.download(filename=output_path)
            return downloaded_file
        except Exception as e:
            print(f"Attempt {retry_count + 1} failed: {str(e)}")
            time.sleep(5)  # wait 5 seconds before retrying
            retry_count += 1
    print("Failed to download after several retries.")
    return None

def convert_time_to_seconds(time):
    if isinstance(time, str) and ':' in time:
        minutes, seconds = map(int, time.split(':'))
        return minutes * 60 + seconds
    elif isinstance(time, (int, float)):
        return time
    else:
        raise ValueError("Time format must be a string 'MM:SS' or a number representing seconds")

def convert_to_ogg(input_path, output_path, start_time=0, end_time=None):
    try:
        # Convert start time to seconds and add offset
        start_total_seconds = convert_time_to_seconds(start_time) + 0.15
        
        # Initialize the ffmpeg command
        command = [
            'ffmpeg', '-ss', str(start_total_seconds), '-i', input_path,
            '-c:a', 'libvorbis', '-q:a', '5'
        ]
        
        if end_time is not None:
            # Convert end time to seconds
            end_total_seconds = convert_time_to_seconds(end_time)
            # Calculate the duration of the clip
            clip_duration = end_total_seconds - start_total_seconds
            command.extend(['-t', str(clip_duration)])
        
        command.append(output_path)
        
        subprocess.run(command, check=True)
        return output_path
    except subprocess.CalledProcessError as e:
        print(f"Error during conversion: {e}")
        return None
    except ValueError as e:
        print(f"Invalid time format: {e}")
        return None

# URLs for YouTube videos
youtube_url1 = 'https://www.youtube.com/watch?v=4WhizSggjfo'
youtube_url2 = 'https://www.youtube.com/watch?v=R-jegU5ehVA&t=11s'

start_time1 = 8.29
start_time2 = parse_start_time_from_url(youtube_url2)
end_time1 = "52:30"
end_time2 = "51:39"

# Download audio files
audio_path1 = download_audio_with_retries(youtube_url1, 'youtube_audio1.mp4')
audio_path2 = download_audio_with_retries(youtube_url2, 'youtube_audio2.mp4')


# Convert to OGG
ogg_path1 = "youtube_audio1.ogg"
ogg_path2 = "youtube_audio2.ogg"
#convert_to_ogg(audio_path1, ogg_path1, start_time1, end_time1)
convert_to_ogg(audio_path2, ogg_path2, start_time2, end_time2)

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

'youtube_audio2.ogg'

In [2]:
import gc

HOP_LENGTH = 512
HOP_LENGTH_1 = 1024
HOP_LENGTH_2 = 256
OVERLAP_HOP = 256

def load_and_preprocess_audio_ORIGINAL(audio_path, target_sr=11025):
    # Load audio file at a reduced sample rate
    y, sr = librosa.load(audio_path, sr=target_sr)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256)
    log_S = librosa.power_to_db(S, ref=np.max)
    return log_S.T, sr

def load_and_preprocess_audio(audio_path, target_sr=11025, hop_length=HOP_LENGTH):
    # Load audio file at a higher sample rate for better temporal resolution
    y, sr = librosa.load(audio_path, sr=target_sr)
    
    # Harmonic-Percussive Source Separation (HPSS) --- cutting for resource savings
    #y_harmonic, y_percussive = librosa.effects.hpss(y)
    
    # Mel-spectrogram for harmonic component
    #S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=64, hop_length=HOP_LENGTH)
    #log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
    
    # Constant-Q Transform for better frequency resolution
    #CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

    # Combine features
    #combined_features = np.vstack((log_S_harmonic, CQT))
    #print(f"Combined features shape: {combined_features.shape}")
    
    #return combined_features.T, sr

    #BELOW IS STRIPPED VERSION OF ABOVE
    y, sr = librosa.load(audio_path, sr=sr)
    
    # Mel-spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=512, hop_length=HOP_LENGTH)
    log_S = librosa.power_to_db(S, ref=np.max)
    
    # Constant-Q Transform
    CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)

    combined_features = np.vstack((log_S, CQT))
    return combined_features.T, sr

    
def load_and_preprocess_audio_INCREMENTAL_NOHPSS(audio_path, target_sr=11025, chunk_duration=10, n_mels=256):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = num_chunks // 10  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Mel-spectrogram
        S = librosa.feature.melspectrogram(y=y_chunk, sr=sr, n_mels=n_mels, hop_length=HOP_LENGTH)
        log_S = librosa.power_to_db(S, ref=np.max)
        
        # Constant-Q Transform
        CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)
        
        # Ensure the same length for both arrays
        min_length = min(log_S.shape[1], CQT.shape[1])
        log_S = log_S[:, :min_length]
        CQT = CQT[:, :min_length]
        
        # Normalize features
        log_S = normalize_features(log_S)
        CQT = normalize_features(CQT)
        
        # Combine features
        combined_chunk_features = np.vstack((log_S, CQT))

        combined_features.append(combined_chunk_features.T)
        del log_S, CQT, y_chunk, S, combined_chunk_features
        gc.collect()
        
        # Print progress
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr
    

def load_and_preprocess_audio_INCREMENTAL(audio_path, target_sr=11025, chunk_duration=10):
    # Load the audio file in chunks
    y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    chunk_length = int(chunk_duration * sr)  # Convert chunk duration to samples
    num_chunks = (len(y) + chunk_length - 1) // chunk_length  # Calculate the number of chunks

    combined_features = []
    progress_step = num_chunks // 10  # Progress step for every 10%
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Harmonic-Percussive Source Separation (HPSS)
        y_harmonic, y_percussive = librosa.effects.hpss(y_chunk)
        
        # Mel-spectrogram for harmonic component
        S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=256)
        log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
        
        # Constant-Q Transform for better frequency resolution
        CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_chunk, sr=sr)), ref=np.max)
        
        # Combine features
        combined_chunk_features = np.vstack((log_S_harmonic, CQT))
        
        combined_features.append(combined_chunk_features.T)
        
        # Print progress
        if (i + 1) % progress_step == 0:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    # Concatenate all chunk features along the time axis
    combined_features = np.concatenate(combined_features, axis=0)
    
    return combined_features, sr

def load_and_preprocess_audio_OVERLAP(audio_path, target_sr=11025, chunk_duration=600, n_mels=256):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)
    num_chunks = (len(y) + chunk_length - 1) // chunk_length

    combined_features = []
    progress_step = max(1, num_chunks // 10)
    
    num_overlaps = HOP_LENGTH // OVERLAP_HOP
    print(f"Creating {num_overlaps} overlapping windows per chunk")
    
    print(f"Processing file: {audio_path}")
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Overlapping windows
        chunk_features = []
        for offset in range(0, HOP_LENGTH, OVERLAP_HOP):
            if offset > 0:
                y_shifted = np.pad(y_chunk, (offset, 0), mode='constant')[:-offset]
            else:
                y_shifted = y_chunk
            
            # HPSS
            y_harmonic, y_percussive = librosa.effects.hpss(y_shifted)
            
            # Mel-spectrogram
            S_harmonic = librosa.feature.melspectrogram(y=y_harmonic, sr=sr, n_mels=n_mels, hop_length=HOP_LENGTH)
            log_S_harmonic = librosa.power_to_db(S_harmonic, ref=np.max)
            
            # Constant-Q Transform
            CQT = librosa.amplitude_to_db(np.abs(librosa.cqt(y_shifted, sr=sr, hop_length=HOP_LENGTH)), ref=np.max)
            
            # Combine features for this offset
            combined_chunk_features = np.vstack((log_S_harmonic, CQT))
            chunk_features.append(combined_chunk_features.T)
            del y_shifted, y_harmonic, y_percussive, S_harmonic, log_S_harmonic, CQT
            gc.collect()

        combined_chunk_features = np.concatenate(chunk_features, axis=1)
        combined_features.append(combined_chunk_features)
        del chunk_features
        gc.collect()
        
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

#low and high averageed hoplengths
def load_and_preprocess_audio_AVERAGE(audio_path, target_sr=11025, chunk_duration=600, n_mels=256):
    y, sr = librosa.load(audio_path, sr=target_sr)
    chunk_length = int(chunk_duration * sr)
    num_chunks = (len(y) + chunk_length - 1) // chunk_length

    combined_features = []
    progress_step = max(1, num_chunks // 10)
    
    num_overlaps = HOP_LENGTH_2 // OVERLAP_HOP
    print(f"Creating {num_overlaps} overlapping windows per chunk")
    
    print(f"Processing file: {audio_path}")
    print(f"Total chunks: {num_chunks}, Progress step: {progress_step}")
    
    for i in range(num_chunks):
        start_sample = i * chunk_length
        end_sample = min((i + 1) * chunk_length, len(y))
        y_chunk = y[start_sample:end_sample]
        
        # Overlapping windows for both hop lengths
        chunk_features_list = []
        max_length = 0
        for hop_length in [HOP_LENGTH_1, HOP_LENGTH_2]:
            chunk_features = []
            for offset in range(0, hop_length, OVERLAP_HOP):
                if offset > 0:
                    y_shifted = np.pad(y_chunk, (offset, 0), mode='constant')[:-offset]
                else:
                    y_shifted = y_chunk
                
                # Mel-spectrogram
                S = librosa.feature.melspectrogram(y=y_shifted, sr=sr, n_mels=n_mels, hop_length=hop_length)
                log_S = librosa.power_to_db(S, ref=np.max)
                
                chunk_features.append(log_S.T)
                del y_shifted, S, log_S
                gc.collect()

            # Find the maximum length of the feature matrices
            max_length = max(max_length, max(f.shape[0] for f in chunk_features))
            chunk_features_list.append(chunk_features)
            del chunk_features
            gc.collect()
        
        # Resize all feature matrices to the maximum length and average
        resized_chunk_features_list = []
        for features in chunk_features_list:
            resized_features = [resize(f, (max_length, f.shape[1]), anti_aliasing=True) for f in features]
            averaged_chunk_features = np.mean(resized_features, axis=0)
            resized_chunk_features_list.append(averaged_chunk_features)
        
        # Average the features from different hop lengths
        combined_chunk_features = np.mean(resized_chunk_features_list, axis=0)
        combined_features.append(combined_chunk_features)
        del chunk_features_list, resized_chunk_features_list
        gc.collect()
        
        if (i + 1) % progress_step == 0 or (i + 1) == num_chunks:
            print(f"Processed {((i + 1) / num_chunks) * 100:.0f}% of chunks")

    combined_features = np.concatenate(combined_features, axis=0)
    print(f"Combined features shape: {combined_features.shape}")
    
    return combined_features, sr

def normalize_features(features, epsilon=1e-8):
    mean = np.mean(features, axis=0)
    std_dev = np.std(features, axis=0)
    return (features - mean) / (std_dev + epsilon)

    
# Load and preprocess both audio recordings in OGG format
S1, sr1 = load_and_preprocess_audio_INCREMENTAL_NOHPSS(ogg_path1)

Total chunks: 315, Progress step: 31
Processed 10% of chunks
Processed 20% of chunks
Processed 30% of chunks
Processed 39% of chunks
Processed 49% of chunks
Processed 59% of chunks
Processed 69% of chunks
Processed 79% of chunks
Processed 89% of chunks
Processed 98% of chunks
Processed 100% of chunks
Combined features shape: (67861, 340)


In [3]:
#making own step so no need to re-run audio1 for multiple recordings.
S2, sr2 = load_and_preprocess_audio_INCREMENTAL_NOHPSS(ogg_path2)

Total chunks: 309, Progress step: 30
Processed 10% of chunks
Processed 19% of chunks
Processed 29% of chunks
Processed 39% of chunks
Processed 49% of chunks
Processed 58% of chunks
Processed 68% of chunks
Processed 78% of chunks
Processed 87% of chunks
Processed 97% of chunks
Processed 100% of chunks
Combined features shape: (66698, 340)


In [4]:
print(S1.shape)
print(S2.shape)

(67861, 340)
(66698, 340)


In [5]:
#from dtw import dtw

#def dynamic_time_warping(S1, S2):
    # Perform normal DTW
    #alignment = dtw(S1, S2)
    #return list(zip(alignment.index1, alignment.index2))

def dynamic_time_warping_approx(S1, S2):
    distance, path = fastdtw(S1, S2)
    return path

# Dynamic Time Warping
warping_path = dynamic_time_warping_approx(S1, S2)
#warping_path = dynamic_time_warping(S1, S2)

In [6]:
def adjust_timestamps(wp, timestamps, sr, offset):
    mapping = {row[0]: row[1] for row in wp}
    adjusted_timestamps = []
    
    for entry in timestamps:
        original_frame = int((entry['t'] + offset) * sr / HOP_LENGTH)
        if original_frame in mapping:
            adjusted_time = mapping[original_frame] * HOP_LENGTH / sr - offset
            print(entry['mix'], original_frame, mapping[original_frame])
            adjusted_timestamps.append({"t": adjusted_time, "mix": entry['mix']})
    
    # Ensure the last timestamp is included and set "t" to 9999
    if timestamps:
        last_entry = timestamps[-1]
        last_entry_adjusted = {"t": 9999, "mix": last_entry['mix']}
        if adjusted_timestamps and adjusted_timestamps[-1]['mix'] == last_entry['mix']:
            adjusted_timestamps[-1] = last_entry_adjusted
        else:
            adjusted_timestamps.append(last_entry_adjusted)
    
    return adjusted_timestamps

# Sample JSON timestamps for the first recording (assumed already loaded)
adjusted_timestamps = adjust_timestamps(warping_path, timestamps1, sr1, start_time1)

0 178 200
1 228 250
2 252 274
3 281 303
4 305 327
5 333 355
6 356 378
7 377 399
8 408 434
9 434 460
10 465 492
11 491 518
12 518 546
13 544 572
14 577 606
15 604 633
16 630 661
17 655 687
18 680 712
19 707 741
20 734 769
21 759 797
22 786 824
23 815 856
24 842 885
25 866 910
26 892 937
27 919 965
28 944 992
29 968 1017
30 994 1045
31 1020 1071
32 1045 1098
33 1069 1122
34 1091 1144
35 1119 1174
36 1143 1199
37 1171 1227
38 1197 1254
39 1221 1278
40 1247 1304
41 1274 1333
42 1300 1359
43 1327 1386
44 1354 1414
45 1379 1439
46 1407 1468
47 1433 1494
48 1461 1523
49 1488 1551
50 1513 1578
51 1539 1604
52 1568 1635
53 1595 1663
54 1622 1691
55 1648 1718
56 1674 1746
57 1704 1776
58 1732 1804
59 1758 1830
60 1782 1854
61 1808 1880
62 1834 1906
63 1859 1931
64 1884 1957
65 1911 1985
66 1936 2010
67 1963 2037
68 1986 2061
69 2014 2090
70 2044 2121
71 2066 2144
72 2091 2171
73 2119 2199
74 2145 2226
75 2173 2255
76 2200 2282
77 2225 2307
78 2250 2330
79 2277 2355
80 2304 2382
81 2330 2408
82 2

In [7]:
def calculate_ratios(timestamps):
    ratios = []
    for i in range(1, len(timestamps)):
        current_ratio = abs(timestamps[i]['t'] - timestamps[i-1]['t']) if timestamps[i-1]['t'] != 0 else 0
        ratios.append(current_ratio)
    return ratios

def compare_and_flag_changes(adjusted_timestamps, original_timestamps, audio_length, neighbor_count=5):
    # Set last timestamp as per new requirement
    adjusted_timestamps[-1]['t'] = audio_length + 1

    # Calculate differences and ratios
    adjusted_ratios = calculate_ratios(adjusted_timestamps)
    original_ratios = calculate_ratios(original_timestamps)

    # Array to hold timestamps that are significantly different
    flagged_timestamps = []

    # Analyze ratios for significant changes
    for i in range(len(adjusted_ratios) - 1):  # Ignore the last timestamp in comparison
        start = max(0, i - neighbor_count)
        end = min(len(original_ratios) - 1, i + neighbor_count + 1)  # Avoid including the last in comparison
        
        # Calculate neighborhood average without including out-of-range values
        neighborhood_original = original_ratios[start:end]
        if not neighborhood_original:
            continue
        neighborhood_average = np.mean(neighborhood_original)
        
        # Check if the current adjusted ratio is significantly different
        if adjusted_ratios[i] > 1.5 * neighborhood_average:
            flagged_timestamps.append({
                "mix": adjusted_timestamps[i]['mix'],
                "original_ratio": original_ratios[i] if i < len(original_ratios) else 0,
                "adjusted_ratio": adjusted_ratios[i],
                "average_neighbors": neighborhood_average
            })

    return flagged_timestamps

# Adjust so that the first timestamp is zero
initial_offset = -adjusted_timestamps[0]['t']

# Initialize an empty list to store the new adjusted timestamps
new_adjusted_timestamps = []
previous_t = None  # Variable to hold the previous timestamp

for item in adjusted_timestamps:
    adjusted_t = round(item["t"] + initial_offset, 3)
    new_adjusted_timestamps.append({"t": adjusted_t, "mix": item["mix"]})
    
    # Check if the previous timestamp is defined and compare the current timestamp with the previous one
    if previous_t is not None and (adjusted_t - previous_t < 0.5):
        difference = adjusted_t - previous_t
        print(f"Close timestamps found: Mix: {item['mix']}, Difference: {difference:.3f}, Previous - {previous_t}, Current - {adjusted_t}")
    
    # Update the previous_t to the current timestamp for the next iteration
    previous_t = adjusted_t


# Print the adjusted timestamps and initial offset
print(json.dumps(new_adjusted_timestamps, indent=4))
# Here we adjust to show the total offset from the original video start
full_offset = abs(initial_offset) + abs(start_time2)
print(f"Total Offset from Video Start: {full_offset}, initial {initial_offset} + start_time2 {start_time2}")

#Flag anything that exceeds 50% difference compared to neighboring measures.
audio_length = librosa.get_duration(path=ogg_path2)
flagged_timestamps = compare_and_flag_changes(new_adjusted_timestamps, timestamps1, audio_length)
print("Flagged Timestamps:")
for ft in flagged_timestamps:
    print(f"Mix: {ft['mix']}, Original Ratio: {ft['original_ratio']:.3f}, Adjusted Ratio: {ft['adjusted_ratio']:.3f}, Neighbors' Avg.: {ft['average_neighbors']:.3f}")

# Cleanup downloaded and converted files
#os.remove(audio_path1)
#os.remove(audio_path2)
#os.remove(ogg_path1)
#os.remove(ogg_path2)

Close timestamps found: Mix: 945, Difference: 0.465, Previous - 1963.99, Current - 1964.455
Close timestamps found: Mix: 951, Difference: 0.465, Previous - 1967.148, Current - 1967.613
Close timestamps found: Mix: 961, Difference: 0.464, Previous - 1972.489, Current - 1972.953
Close timestamps found: Mix: 967, Difference: 0.464, Previous - 1975.693, Current - 1976.157
Close timestamps found: Mix: 980, Difference: 0.465, Previous - 1982.705, Current - 1983.17
Close timestamps found: Mix: 1002, Difference: 0.465, Previous - 1994.826, Current - 1995.291
Close timestamps found: Mix: 1008, Difference: 0.417, Previous - 1998.124, Current - 1998.541
Close timestamps found: Mix: 1026, Difference: 0.465, Previous - 2007.829, Current - 2008.294
Close timestamps found: Mix: 1044, Difference: 0.465, Previous - 2017.628, Current - 2018.093
Close timestamps found: Mix: 1047, Difference: 0.464, Previous - 2019.347, Current - 2019.811
Close timestamps found: Mix: 1065, Difference: 0.464, Previous - 20